# Bài 22 · Tiền huấn luyện & tinh chỉnh: học trước trên dữ liệu không nhãn, rồi chỉnh một chút cho việc cần làm

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/22-tien-huan-luyen.ipynb)

Notebook đi kèm [Bài 22 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/tien-huan-luyen/) (demo tương tác nằm ở trang bài học).
Ta dùng lại MLP MNIST của khóa học (784 → 256 → 128, đúng bộ mã hóa của Bài 19) và hỏi ba câu:
**tiền huấn luyện không nhãn giúp được bao nhiêu, và khi nào?**, **một thăm dò tuyến tính đo được gì?** và
**LoRA học ít tham số hơn thì mất gì, giữ được gì?** Cuối bài, ta đọc các số đo của một mô hình CLIP thật.

**Bạn sẽ làm:**
1. Tạo nhãn từ chính dữ liệu (token tiếp theo, token bị che), tự viết mất mát NT-Xent của SimCLR (có kiểm tra bằng
   sai phân hữu hạn), chạy một SimCLR nhỏ trên ảnh không nhãn, rồi đo bộ mã hóa của nó bằng một thăm dò tuyến tính.
2. Đọc kết quả quét số nhãn (10–55.000, 3 seed) và chạy lại các mốc so sánh: mạng ngẫu nhiên, điểm ảnh thô.
3. Đọc các số đo zero-shot và thăm dò tuyến tính của CLIP ViT-B/32 trên ảnh thật và trên MNIST.
4. Viết LoRA $h = xW + \tfrac{\alpha}{r}\,xAB$, chuyển một mạng từ chữ số 0–4 sang 5–9 bằng thăm dò, LoRA và tinh
   chỉnh toàn bộ, rồi đo mức quên việc cũ.
5. Đo hạng của cập nhật $\Delta W$, dựng lại mạng của demo với $k$ hướng lớn nhất của nó, và tính số tham số LoRA
   của GPT-3.

**Cần biết trước:** [Bài 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/transformer/),
[Bài 18 · Vision Transformer](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/vision-transformer/) (tháp ảnh của CLIP
là một ViT), [Bài 19 · Autoencoder & VAE](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/autoencoder-vae/) và bộ tối ưu
Adam của [Bài 07](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/bo-toi-uu/). **Thời gian chạy:** dưới 1 phút trên
CPU, kể cả tải MNIST (~11 MB) và 1,1 MB kết quả của demo lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra, hoặc đọc từ đúng các tệp mà demo dùng, và trùng với
số trong demo. Các lần chạy đầy đủ (9 cách học × 5 mức nhãn × 3 seed, một lưới tốc độ học cho mỗi ô, và CLIP chạy trên
12.500 ảnh MNIST) mất hàng chục phút trên máy nhiều lõi, nên notebook nạp kết quả của chúng và chạy lại một phiên bản nhỏ
của từng cơ chế. CLIP có 151 triệu tham số; notebook không tải trọng số của nó, chỉ đọc các cosin đã đo.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.
`span()` in khoảng min–max qua 3 seed, như demo.

In [ ]:
import base64
import gzip
import hashlib
import io
import itertools
import json
import math
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng điểm, nét, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("cam_xam_xanh", [ORANGE, "#f2f2f2", BLUE])  # âm → 0 → dương
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def span(values, digits=2):
    """Khoảng min–max (tỉ lệ → phần trăm) qua các seed: 80,63–82,04%."""
    lo, hi = vn(100 * min(values), digits), vn(100 * max(values), digits)
    return f"{lo}%" if lo == hi else f"{lo}–{hi}%"


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 60.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def log_tick(v, _pos=None):
    """Nhãn vạch trục log: 0,5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Đổi nhãn mọi trục của hình (cả thanh màu) sang kiểu Việt Nam; gọi ngay trước plt.show().
    Trục log lấy vạch 1–2–5 nếu ngắn, hoặc các lũy thừa của 10 (cách đều, tối đa ~9 vạch) nếu dài."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def sci(x, digits=1):
    """Số rất nhỏ: 1,9·10⁻⁷."""
    e = math.floor(math.log10(abs(x)))
    return f"{vn(x / 10**e, digits)}·10{str(e).translate(SUP)}"


def show_digits(axes, images, titles=None, fontsize=9):
    """Vẽ mỗi ảnh 784 điểm (giá trị 0–1) vào một ô, nền trắng, nét đen."""
    for i, ax in enumerate(np.ravel(axes)):
        ax.axis("off")
        if i < len(images):
            ax.imshow(np.asarray(images[i]).reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
            if titles is not None:
                ax.set_title(titles[i], fontsize=fontsize)


print(vn(13007, 0), vn(0.1, 1), vn(-4.3805, 4), pct(0.993527), span([0.8063, 0.8204]), sci(1.9e-7))

## 2 · Dữ liệu

MNIST (LeCun và cộng sự, 1998): 60.000 ảnh huấn luyện và 10.000 ảnh kiểm tra, chữ số viết tay xám 28×28. Ô dưới tải
4 tệp gốc và kiểm tra mã MD5, nên ai chạy cũng có đúng cùng một dữ liệu. Giống các lần chạy của demo, ta chia:
- **kho ảnh huấn luyện:** 55.000 ảnh đầu; SimCLR học trên cả kho mà *không nhìn nhãn*, còn các tập có nhãn được rút
  từ đây;
- **kiểm định:** 5.000 ảnh cuối của tệp huấn luyện (demo chọn tốc độ học $\eta$ ở đây, không bao giờ trên tập kiểm tra);
- **kiểm tra:** 10.000 ảnh của tệp kiểm tra, chỉ để báo kết quả.

Ô cũng nạp bốn tệp kết quả mà demo dùng (`fetch_json`, có kiểm tra MD5):
- `pretrain-label.json`: 9 cách học × 5 mức nhãn × 3 seed (thẻ 1 của demo);
- `pretrain-transfer.json`: chuyển từ 0–4 sang 5–9 bằng thăm dò, LoRA hạng 1–16, tinh chỉnh toàn bộ hoặc học từ đầu;
- `pretrain-transfer-weights.json`: mạng tinh chỉnh toàn bộ của seed 0 mà demo dựng lại trong trình duyệt;
- `pretrain-clip.json`: các số đo của CLIP ViT-B/32.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # mã MD5 chuẩn của MNIST
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 784)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# Kết quả của các lần huấn luyện dài (cùng tệp với demo trên web). Khi xuất bản, build.py thay chuỗi giữ chỗ trong
# urlretrieve dưới đây bằng thư mục data/ của repo notebook công khai.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
LAB = fetch_json("pretrain-label.json", "7ff752f72033d5a8d2605c0ac5db1b80")
TRA = fetch_json("pretrain-transfer.json", "c2c712a895375ef0161d20797359b330")
SHIP = fetch_json("pretrain-transfer-weights.json", "906cf393e1eae9413d3f41563da66b6e")
CLIP = fetch_json("pretrain-clip.json", "c0f381b32aed08626b4b4b8c17090f4f")
size = sum(f.stat().st_size for f in Path("data").glob("pretrain-*.json"))
print(
    f"huấn luyện: {vn(len(train_img), 0)} ảnh · kiểm tra: {vn(len(test_img), 0)} ảnh · 4 tệp của demo: {vn(size / 1e6, 1)} MB"
)

Ảnh thành vectơ hàng 784 số trong $[0, 1]$, kiểu float32 như các lần chạy của demo. `subset()` rút một tập có nhãn
cân bằng (cùng số ảnh cho mỗi lớp) từ kho 55.000 ảnh, đúng cách của demo: mỗi seed một lần rút cố định. Với
“55.000 nhãn”, mỗi lớp lấy tối đa 5.500 ảnh, nên lớp nào ít hơn thì lấy hết; tổng thực ra hơi dưới 55.000.

In [ ]:
F32 = np.float32
X, y = train_img.astype(F32) / 255, train_lab.astype(np.int64)
Xt, yt = test_img.astype(F32) / 255, test_lab.astype(np.int64)
POOL, VAL = np.arange(55000), np.arange(55000, 60000)  # kho ảnh huấn luyện · kiểm định


def subset(per_class, seed, classes=range(10)):
    """per_class ảnh mỗi lớp từ kho, một lần rút cố định cho mỗi seed (lấy hết nếu lớp có ít hơn)."""
    rng = np.random.default_rng(40_000 + seed)
    out = []
    for c in classes:
        idx = POOL[y[POOL] == c]
        out.append(rng.choice(idx, per_class, replace=False) if per_class < len(idx) else idx)
    return np.sort(np.concatenate(out))


print(f"kho {vn(len(POOL), 0)} ảnh · kiểm định {vn(len(VAL), 0)} · kiểm tra {vn(len(yt), 0)}")
print("số ảnh mỗi lớp trong kho:", np.bincount(y[POOL]).tolist(), "· mỗi lớp lấy tối đa", vn(5500, 0))
print(f"tập “100 nhãn”: {len(subset(10, 0))} ảnh · tập “55.000 nhãn”: {vn(len(subset(5500, 0)), 0)} ảnh")

## 3 · Nhãn có sẵn trong dữ liệu

Nhãn do con người gán thì đắt và ít; dữ liệu thô thì gần như vô tận. **Tự giám sát** (self-supervised) tự tạo nhãn
từ chính dữ liệu, rồi học bằng cross-entropy quen thuộc của Bài 04:
- **GPT** đoán token tiếp theo: mỗi vị trí $t$ của một câu là một ví dụ, ngữ cảnh $x_{<t}$ và nhãn $x_t$, với mất mát
  $\mathcal L_{\text{LM}} = -\sum_t \log p_\theta(x_t \mid x_{<t})$ (Radford và cộng sự, 2018).
- **BERT** che 15% token rồi đoán lại chúng. Trong số token được chọn, 80% thay bằng `[MASK]`, 10% bằng một token
  ngẫu nhiên và 10% giữ nguyên: lúc tinh chỉnh không còn `[MASK]` nào, nên mô hình không được phép chỉ chú ý tới nó
  (Devlin và cộng sự, 2018).

Ô dưới tạo cả hai loại nhãn từ một câu, không cần ai gán gì. Ở đây một token là một từ (Bài 14 dùng BPE).

In [ ]:
text = (
    "nhãn do con người gán thì đắt và ít , còn dữ liệu thô thì gần như vô tận . tiền huấn luyện tự giám sát tự tạo "
    "nhãn từ chính dữ liệu : đoán token tiếp theo , hoặc đoán lại những token bị che"
)
tokens = text.split()
for t in range(1, 5):  # GPT: ngữ cảnh x_<t → nhãn x_t
    print(f"{' '.join(tokens[:t]):>18}  →  {tokens[t]}")
print(f"… một câu {len(tokens)} token cho {len(tokens) - 1} cặp (ngữ cảnh, nhãn)\n")


SHARE, P_MASK, P_RAND = 0.15, 0.8, 0.1  # công thức của BERT (Devlin và cộng sự, 2018)


def bert_mask(tokens, rng, share=SHARE):
    """Chọn 15% vị trí; mỗi vị trí: 80% → [MASK], 10% → một token ngẫu nhiên, 10% giữ nguyên. Nhãn là token gốc."""
    pos = np.sort(rng.choice(len(tokens), max(1, round(share * len(tokens))), replace=False))
    out, vocab = list(tokens), sorted(set(tokens))
    for i in pos:
        u = rng.random()
        out[i] = "[MASK]" if u < P_MASK else vocab[rng.integers(len(vocab))] if u < P_MASK + P_RAND else tokens[i]
    return out, pos


print(f"BERT chọn {pct(SHARE, 0)} token; trong đó {pct(P_MASK, 0)} → [MASK], {pct(P_RAND, 0)} → token ngẫu nhiên, "
      f"{pct(1 - P_MASK - P_RAND, 0)} giữ nguyên")  # fmt: skip
masked, pos = bert_mask(tokens, np.random.default_rng(1))
print(" ".join(masked))
print("nhãn cần đoán:", {int(i): tokens[i] for i in pos})
# với rất nhiều token, tỉ lệ 80/10/10 hiện ra
big = ["a", "b", "c", "d"] * 25_000
m_big, pos_big = bert_mask(big, np.random.default_rng(0))
kind = np.array(["mask" if m_big[i] == "[MASK]" else "giữ" if m_big[i] == big[i] else "đổi" for i in pos_big])
print(f"kỳ vọng với 4 token: giữ {pct(1 - P_MASK - P_RAND + P_RAND / 4, 1)}, đổi {pct(P_RAND * 3 / 4, 1)}")
print(
    f"{vn(len(pos_big), 0)} vị trí được chọn trên {vn(len(big), 0)} token:",
    {k: pct((kind == k).mean(), 1) for k in ("mask", "đổi", "giữ")},
)

Ở 100.000 token, phần “giữ” khoảng 12,5% chứ không phải 10%: từ vựng ở đây chỉ có 4 token, nên “token ngẫu nhiên”
trúng đúng token cũ 1 lần trong 4, và một phần tư của 10% “đổi” thành “giữ”.

Ảnh không có “token tiếp theo”, nhưng có cách tạo nhãn khác: hai phiên bản biến đổi của cùng một ảnh phải có biểu
diễn giống nhau. Đó là SimCLR (mục 5). Trước đó ta dựng mạng của bài.

## 4 · Mạng của bài, và ba cách học trên nó

Mạng là MLP MNIST của khóa học: bộ mã hóa 784 → 256 → 128 (ReLU, khởi tạo He như Bài 09; đúng bộ mã hóa của Bài 19)
và một lớp đầu ra tuyến tính, theo quy ước hàng:

$$h_1 = \mathrm{ReLU}(xW_1 + b_1), \qquad h_2 = \mathrm{ReLU}(h_1W_2 + b_2), \qquad z = h_2W_h + b_h$$

Ba cách học dùng cùng mạng này, chỉ khác ở **những tham số được học**:
- **thăm dò tuyến tính** (linear probe): đóng băng bộ mã hóa, chỉ học $W_h, b_h$;
- **tinh chỉnh toàn bộ** (full fine-tuning): học mọi tham số, bắt đầu từ bộ mã hóa đã tiền huấn luyện;
- **học từ đầu**: như tinh chỉnh toàn bộ, nhưng bộ mã hóa bắt đầu từ khởi tạo ngẫu nhiên.

Hồi quy softmax trên điểm ảnh thô là trường hợp không có bộ mã hóa: $z = xW_h + b_h$. Các hàm dưới chép đúng cách
demo khởi tạo (mỗi phần một seed riêng: bộ mã hóa `seed`, lớp đầu ra `10.000 + seed`), nên cùng seed cho cùng mạng.

In [ ]:
SIZES = [784, 256, 128]


def init_encoder(seed):
    """Bộ mã hóa 784 → 256 → 128, khởi tạo He: W ~ N(0, 2/d_vào), b = 0 (float32, như demo)."""
    rng, p = np.random.default_rng(seed), {}
    for i, (a, b) in enumerate(itertools.pairwise(SIZES)):
        p[f"W{i}"] = (rng.standard_normal((a, b)) * np.sqrt(2 / a)).astype(F32)
        p[f"b{i}"] = np.zeros(b, F32)
    return p


def add_head(p, k, seed):
    """Một lớp đầu ra k lớp mới: N(0, 1/128) trên bộ mã hóa; toàn 0 cho hồi quy softmax trên điểm ảnh."""
    rng, q = np.random.default_rng(10_000 + seed), dict(p)
    q["Wh"] = (rng.standard_normal((128, k)) * np.sqrt(1 / 128)).astype(F32) if p else np.zeros((784, k), F32)
    q["bh"] = np.zeros(k, F32)
    return q


def depth(p):
    return sum(1 for k in p if k[0] == "W" and k[1:].isdigit())


def softmax(z):
    e = np.exp(z - z.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def forward(p, x):
    """Các lớp ẩn [x, h₁, h₂] và logit z. Nếu p có thêm Aᵢ, Bᵢ (LoRA, mục 10), lớp i cộng thêm (hAᵢ)Bᵢ."""
    hs = [x]
    for i in range(depth(p)):
        a = hs[-1] @ p[f"W{i}"] + p[f"b{i}"]
        if f"A{i}" in p:
            a = a + (hs[-1] @ p[f"A{i}"]) @ p[f"B{i}"]
        hs.append(np.maximum(a, 0))
    return hs, hs[-1] @ p["Wh"] + p["bh"]


def features(p, X, bs=5000):
    """Đầu ra h₂ của bộ mã hóa (đóng băng) cho cả tập; chính điểm ảnh nếu không có bộ mã hóa."""
    if depth(p) == 0:
        return X
    out = []
    for i in range(0, len(X), bs):
        h = X[i : i + bs]
        for j in range(depth(p)):
            h = np.maximum(h @ p[f"W{j}"] + p[f"b{j}"], 0)
        out.append(h)
    return np.concatenate(out)


def acc(p, X, y, off=0, bs=5000):
    """Tỉ lệ đoán đúng; off = 5 khi lớp đầu ra 5 lớp đọc các chữ số 5–9 (mục 10)."""
    pred = np.concatenate([forward(p, X[i : i + bs])[1].argmax(1) for i in range(0, len(X), bs)])
    return float((pred + off == y).mean())

Huấn luyện dùng đúng công thức của demo: Adam (Bài 07), lịch cosin về 0, lô 64 ảnh theo một thứ tự xáo trộn lại sau
mỗi lượt, cross-entropy của Bài 04 và lan truyền ngược của Bài 06. Tham số nào không nằm trong `train` thì đứng yên.
Một thăm dò tuyến tính tính trước đặc trưng $h_2$ của mọi ảnh một lần, rồi chỉ học hồi quy softmax trên chúng.

In [ ]:
class Adam:
    """Adam (Bài 07) cho các tham số có tên trong keys; sửa p tại chỗ."""

    def __init__(self, p, keys, b1=0.9, b2=0.999, eps=1e-8):
        self.m = {k: np.zeros_like(p[k]) for k in keys}
        self.v = {k: np.zeros_like(p[k]) for k in keys}
        self.t, self.b1, self.b2, self.eps = 0, b1, b2, eps

    def step(self, p, g, lr):
        self.t += 1
        c1, c2 = 1 - self.b1**self.t, 1 - self.b2**self.t
        for k in self.m:
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * g[k]
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * g[k] * g[k]
            p[k] = (p[k] - lr * (self.m[k] / c1) / (np.sqrt(self.v[k] / c2) + self.eps)).astype(p[k].dtype)


def lr_at(t, lr, steps, warm=0):
    """Warmup tuyến tính trong warm bước đầu (nếu có), rồi lịch cosin về 0."""
    if t < warm:
        return lr * (t + 1) / warm
    return lr * 0.5 * (1 + np.cos(np.pi * (t - warm) / max(1, steps - warm)))


def grads(p, x, y, train):
    """Cross-entropy và gradient của nó (lan truyền ngược, Bài 06); lớp ẩn chỉ tính khi train có tham số của chúng."""
    hs, z = forward(p, x)
    n = len(y)
    dz = softmax(z)
    loss = float(-np.log(dz[np.arange(n), y] + 1e-12).mean())
    dz[np.arange(n), y] -= 1
    dz /= n
    g = {"Wh": hs[-1].T @ dz, "bh": dz.sum(0)}
    if set(train) - {"Wh", "bh"}:
        dh = dz @ p["Wh"].T
        for i in reversed(range(depth(p))):
            da = dh * (hs[i + 1] > 0)  # ReLU cho gradient đi qua nơi nó dương
            if f"W{i}" in train:
                g[f"W{i}"], g[f"b{i}"] = hs[i].T @ da, da.sum(0)
            if f"A{i}" in p:  # LoRA (mục 10): a = hW + b + (hA)B
                g[f"B{i}"] = (hs[i] @ p[f"A{i}"]).T @ da
                dt = da @ p[f"B{i}"].T
                g[f"A{i}"] = hs[i].T @ dt
            if i > 0:
                dh = da @ p[f"W{i}"].T + (dt @ p[f"A{i}"].T if f"A{i}" in p else 0)
    return loss, g


def fit(p, X, y, train, lr, steps, seed):
    """Adam + cosin trên (X, y), lô 64, như demo. Thăm dò tuyến tính: học trên đặc trưng đã tính trước."""
    rng = np.random.default_rng(30_000 + seed)
    probe = set(train) <= {"Wh", "bh"}
    q = {"Wh": p["Wh"], "bh": p["bh"]} if probe else p
    if probe:
        X = features(p, X)
    opt, n = Adam(q, train), len(X)
    bsz = min(64, n)
    order, pos = rng.permutation(n), 0
    for t in range(steps):
        if pos + bsz > n:  # hết một lượt: xáo trộn lại
            order, pos = rng.permutation(n), 0
        idx = order[pos : pos + bsz]
        pos += bsz
        opt.step(q, grads(q, X[idx], y[idx], train)[1], lr_at(t, lr, steps))
    if probe:
        p["Wh"], p["bh"] = q["Wh"], q["bh"]
    return p


def steps_for(n):
    """Như demo: ít nhất 2.000 bước và ít nhất 5 lượt qua tập có nhãn."""
    return max(2000, math.ceil(5 * n / 64))


print("số bước cho 100 nhãn:", steps_for(100), "· cho tập “55.000 nhãn”:", vn(steps_for(len(subset(5500, 0))), 0))

## 5 · SimCLR: hai phiên bản của cùng một ảnh

SimCLR (Chen và cộng sự, 2020) tạo nhãn bằng phép biến đổi ngẫu nhiên. Lấy một lô $B$ ảnh, biến đổi mỗi ảnh hai lần
thành $2B$ **phiên bản** (view). Hai phiên bản của cùng một ảnh là một **cặp dương**; mọi cặp khác trong lô là cặp âm.
Mạng phải nhận ra cặp dương của mỗi phiên bản giữa $2B - 1$ ứng viên. Nó không biết ảnh nào là chữ số mấy.

Phép biến đổi quyết định mạng học được gì. Với chữ số, demo dùng phép biến đổi affine ngẫu nhiên: xoay ±15°, co giãn
0,85–1,15 lần và dịch ±3 điểm ảnh, nội suy song tuyến tính, nền 0. Ô dưới viết nó bằng numpy (`augment` của demo dùng
`scipy.ndimage.map_coordinates`, cùng kết quả) và vẽ 6 phiên bản của 4 chữ số.

In [ ]:
def augment(X, rng, rot=15.0, scale=(0.85, 1.15), shift=3.0):
    """Một phiên bản affine ngẫu nhiên của mỗi ảnh (B, 784): với mỗi điểm ảnh đích, tìm điểm nguồn rồi nội suy."""
    B = len(X)
    img = X.reshape(B, 28, 28)
    th = np.deg2rad(rng.uniform(-rot, rot, B))
    sc = rng.uniform(scale[0], scale[1], B)
    sh = rng.uniform(-shift, shift, (B, 2))
    yy, xx = np.meshgrid(np.arange(28) - 13.5, np.arange(28) - 13.5, indexing="ij")  # tọa độ quanh tâm ảnh
    c, s = (np.cos(th) / sc)[:, None, None], (np.sin(th) / sc)[:, None, None]
    sy = c * yy - s * xx + 13.5 - sh[:, 0, None, None]  # điểm nguồn (hàng, cột) của từng điểm đích
    sx = s * yy + c * xx + 13.5 - sh[:, 1, None, None]
    inside = (sy >= 0) & (sy <= 27) & (sx >= 0) & (sx <= 27)
    y0, x0 = np.clip(np.floor(sy), 0, 26).astype(int), np.clip(np.floor(sx), 0, 26).astype(int)
    fy, fx, b = sy - y0, sx - x0, np.arange(B)[:, None, None]
    v = (img[b, y0, x0] * (1 - fy) * (1 - fx) + img[b, y0 + 1, x0] * fy * (1 - fx)
         + img[b, y0, x0 + 1] * (1 - fy) * fx + img[b, y0 + 1, x0 + 1] * fy * fx)  # fmt: skip
    return np.where(inside, v, 0).reshape(B, 784).astype(X.dtype)


SIM = LAB["setup"]["simclr"]
assert (SIM["rot"], SIM["scale"], SIM["shift"]) == (15.0, [0.85, 1.15], 3.0)  # đúng phép biến đổi của demo
print(
    f"xoay ±{vn(SIM['rot'], 0)}°, co giãn {vn(SIM['scale'][0])}–{vn(SIM['scale'][1])} lần, dịch ±{vn(SIM['shift'], 0)} điểm ảnh"
)
views_rng = np.random.default_rng(3)
fig, axes = plt.subplots(4, 7, figsize=(7.5, 4.6))
for row, i in zip(axes, [0, 1, 2, 7]):
    show_digits(row[:1], X[i : i + 1], ["ảnh gốc"])
    show_digits(row[1:], augment(np.repeat(X[i : i + 1], 6, 0), views_rng), ["phiên bản"] * 6)
plt.tight_layout()
plt.show()

Cột đầu là ảnh gốc, 6 cột sau là các phiên bản ngẫu nhiên của nó: nghiêng, to nhỏ, lệch khác nhau, nhưng vẫn là cùng
chữ số. Nếu biểu diễn của mạng giống nhau cho mọi phiên bản của một ảnh và khác nhau giữa các ảnh, mạng đã phải học
những gì không đổi khi ảnh bị xoay, co giãn hay dịch.

### Mất mát NT-Xent

Mỗi phiên bản đi qua bộ mã hóa rồi một **đầu chiếu** (projection head) 128 → 128 → 64, cho vectơ $z$. Gọi
$s_{ij} = \cos(z_i, z_j)$ là cosin giữa hai phiên bản. Với phiên bản $i$ có cặp dương $p(i)$, mất mát là cross-entropy
của bài toán “chọn đúng cặp dương” với nhiệt độ $\tau$ (demo: 0,5):

$$\ell_i = -\log \frac{\exp(s_{i,p(i)}/\tau)}{\sum_{k \ne i} \exp(s_{ik}/\tau)}, \qquad
\mathcal L = \frac{1}{2B}\sum_{i=1}^{2B} \ell_i$$

Viết bằng vòng lặp cho một phiên bản trước, rồi cả lô một lần. Phiên bản $i$ và $i + B$ là một cặp.

In [ ]:
TAU = 0.5


def ntxent_loop(z, i, tau=TAU):
    """ℓᵢ cho một phiên bản i, bằng vòng lặp qua mọi phiên bản k ≠ i."""
    B = len(z) // 2
    u = z / np.linalg.norm(z, axis=1, keepdims=True)
    pos = i + B if i < B else i - B
    denom = sum(np.exp(u[i] @ u[k] / tau) for k in range(len(z)) if k != i)
    return -np.log(np.exp(u[i] @ u[pos] / tau) / denom)


def ntxent(z, tau=TAU):
    """NT-Xent trên 2B phiên bản: mất mát, ∂L/∂z, và tỉ lệ phiên bản có láng giềng gần nhất là cặp dương của nó."""
    n = len(z)
    nz = np.linalg.norm(z, axis=1, keepdims=True) + 1e-12
    u = z / nz
    S = (u @ u.T) / tau
    np.fill_diagonal(S, -np.inf)  # k ≠ i
    pos = np.concatenate([np.arange(n // 2, n), np.arange(n // 2)])
    m = S.max(1, keepdims=True)
    e = np.exp(S - m)
    loss = float((np.log(e.sum(1)) + m[:, 0] - S[np.arange(n), pos]).mean())
    dS = e / e.sum(1, keepdims=True)  # softmax theo hàng, trừ one-hot của cặp dương (Bài 04)
    dS[np.arange(n), pos] -= 1
    dS /= n
    dU = (dS + dS.T) @ u / tau  # S = UUᵀ/τ đối xứng: mỗi sᵢⱼ góp vào cả hàng i lẫn hàng j
    dz = (dU - u * (u * dU).sum(1, keepdims=True)) / nz  # qua phép chuẩn hóa u = z/‖z‖
    return loss, dz, float((S.argmax(1) == pos).mean())


z_toy = np.random.default_rng(0).standard_normal((8, 5))  # B = 4 ảnh → 8 phiên bản, z 5 chiều
loss_loop = np.mean([ntxent_loop(z_toy, i) for i in range(8)])
loss_vec, dz_toy, _ = ntxent(z_toy)
print(f"vòng lặp: {vn(loss_loop, 6)} · cả lô: {vn(loss_vec, 6)} · đoán bừa: log 7 = {vn(np.log(7), 6)}")
assert np.isclose(loss_loop, loss_vec)

Hai cách cho cùng một số. Nếu mọi $z$ giống hệt nhau, mỗi phiên bản thấy 7 ứng viên như nhau và $\ell_i = \log 7$.
Với $z$ ngẫu nhiên, cặp dương không gần nhau hơn cặp âm, nên mất mát cùng cỡ đó; ở đây còn cao hơn, vì có cặp âm
tình cờ gần nhau hơn cặp dương. Học là kéo mất mát xuống dưới mốc đó.

Gradient viết tay trong `ntxent` đi qua ba bước: softmax theo hàng (như Bài 04), ma trận cosin đối xứng
$S = UU^\top/\tau$, và phép chuẩn hóa $u = z/\lVert z\rVert$. Kiểm tra nó bằng sai phân hữu hạn trung tâm trên từng
phần tử của $z$:

In [ ]:
H = 1e-6
num = np.zeros_like(z_toy)
for idx in np.ndindex(*z_toy.shape):
    zp, zm = z_toy.copy(), z_toy.copy()
    zp[idx] += H
    zm[idx] -= H
    num[idx] = (ntxent(zp)[0] - ntxent(zm)[0]) / (2 * H)
rel = np.abs(num - dz_toy).max() / np.abs(num).max()
print(f"sai số tương đối lớn nhất trên {z_toy.size} phần tử: {sci(rel)}")
assert rel < 1e-6

## 6 · Một SimCLR nhỏ, rồi đo bằng thăm dò tuyến tính

Bộ mã hóa của SimCLR là đúng bộ mã hóa 784 → 256 → 128 của mục 4; đầu chiếu chỉ dùng lúc tiền huấn luyện rồi bỏ đi.
Lan truyền ngược nối gradient $\partial\mathcal L/\partial z$ của `ntxent` qua đầu chiếu và bộ mã hóa.

Demo tiền huấn luyện 5.000 bước, mỗi bước 256 ảnh (512 phiên bản). Ở đây ta chạy 200 bước với 128 ảnh, cùng khởi tạo
(seed 0), cùng Adam $\eta = 10^{-3}$ với warmup 2% rồi cosin. Không có nhãn nào được dùng. Để thấy mạng học được gì,
ta đo tỉ lệ cặp dương tìm đúng trên 256 ảnh kiểm định (512 phiên bản) trước và sau khi học.

In [ ]:
def sim_init(seed):
    """Bộ mã hóa của mục 4 (cùng seed) + đầu chiếu 128 → ReLU 128 → 64."""
    p = init_encoder(seed)
    rng = np.random.default_rng(50_000 + seed)
    p["P0"] = (rng.standard_normal((128, 128)) * np.sqrt(2 / 128)).astype(F32)
    p["c0"] = np.zeros(128, F32)
    p["P1"] = (rng.standard_normal((128, 64)) * np.sqrt(1 / 128)).astype(F32)
    p["c1"] = np.zeros(64, F32)
    return p


def sim_forward(p, x):
    h1 = np.maximum(x @ p["W0"] + p["b0"], 0)
    h2 = np.maximum(h1 @ p["W1"] + p["b1"], 0)
    g1 = np.maximum(h2 @ p["P0"] + p["c0"], 0)
    return (x, h1, h2, g1), g1 @ p["P1"] + p["c1"]


def sim_grads(p, x):
    (x0, h1, h2, g1), z = sim_forward(p, x)
    loss, dz, hit = ntxent(z)
    g = {"P1": g1.T @ dz, "c1": dz.sum(0)}
    d = (dz @ p["P1"].T) * (g1 > 0)
    g["P0"], g["c0"] = h2.T @ d, d.sum(0)
    d = (d @ p["P0"].T) * (h2 > 0)
    g["W1"], g["b1"] = h1.T @ d, d.sum(0)
    d = (d @ p["W1"].T) * (h1 > 0)
    g["W0"], g["b0"] = x0.T @ d, d.sum(0)
    return loss, g, hit


def simclr(steps, batch, lr=1e-3, seed=0):
    """SimCLR như demo: mỗi bước một lô ảnh không nhãn từ kho, hai phiên bản mỗi ảnh, Adam + warmup 2% + cosin."""
    p = sim_init(seed)
    opt, rng = Adam(p, list(p)), np.random.default_rng(60_000 + seed)
    for t in range(steps):
        xb = X[rng.integers(0, len(POOL), batch)]
        _, g, _ = sim_grads(p, np.concatenate([augment(xb, rng), augment(xb, rng)]))
        opt.step(p, g, lr_at(t, lr, steps, warm=max(1, steps // 50)))
    return p


ev_rng = np.random.default_rng(1)
ev = np.concatenate([augment(X[VAL[:256]], ev_rng), augment(X[VAL[:256]], ev_rng)])  # 256 cặp dương
sim0 = sim_init(0)
sim = simclr(steps=200, batch=128)
for name, z in [
    ("điểm ảnh thô", ev),
    ("mạng ngẫu nhiên", sim_forward(sim0, ev)[1]),
    ("sau 200 bước", sim_forward(sim, ev)[1]),
]:
    loss, _, hit = ntxent(z)
    print(f"{name:>15}: NT-Xent {vn(loss, 3)} · cặp dương tìm đúng {pct(hit, 1)} (đoán bừa: {pct(1 / 511, 1)})")
demo_sim = SIM["seeds"]
print(
    f"demo, 5.000 bước × 256 ảnh: NT-Xent {vn(demo_sim[0]['loss'], 3)}, cặp dương đúng {span([s['posAcc'] for s in demo_sim], 1)} (3 seed, trên lô huấn luyện)"
)
assert ntxent(sim_forward(sim, ev)[1])[2] > 2 * ntxent(sim_forward(sim0, ev)[1])[2]

Ngay cả cosin giữa hai ảnh điểm ảnh thô cũng tìm đúng cặp dương một phần nhỏ, vì một chữ số lệch vài điểm ảnh vẫn
còn chồng lên chính nó. Mạng ngẫu nhiên còn kém hơn thế. 200 bước SimCLR đã đưa tỉ lệ lên vài lần; 5.000 bước của demo
lên 95,2–95,6%.

**Thăm dò tuyến tính** đặt câu hỏi khác: đóng băng bộ mã hóa, chỉ học lớp đầu ra từ 100 ảnh có nhãn (10 mỗi lớp), thì
đúng bao nhiêu trên 10.000 ảnh kiểm tra? Ta so ba bộ mã hóa: không có (hồi quy softmax trên điểm ảnh thô), mạng ngẫu
nhiên (khởi tạo seed 0, chưa học gì) và chính mạng đó sau 200 bước SimCLR. Với hai mốc đầu, ta dùng $\eta$ mà demo đã
chọn trên tập kiểm định; với SimCLR nhỏ, ta chọn $\eta$ trên tập kiểm định như demo. Không ảnh kiểm tra nào được dùng
để chọn.

In [ ]:
def probe(enc, per_class, seed=0, lrs=None, method=None):
    """Thăm dò tuyến tính trên bộ mã hóa enc (đóng băng); η chọn trên kiểm định. Trả về (test, val, η)."""
    idx = subset(per_class, seed)
    n = str(10 * per_class) if per_class < 5500 else "55000"
    if lrs is None:  # η mà demo đã chọn cho đúng ô này
        lrs = [next(r["lr"] for r in LAB["methods"][method]["runs"][n] if r["seed"] == seed)]
    best = None
    for lr in lrs:
        p = fit(add_head({k: enc[k] for k in ("W0", "b0", "W1", "b1") if k in enc}, 10, seed), X[idx], y[idx],
                ["Wh", "bh"], lr, steps_for(len(idx)), seed)  # fmt: skip
        val = acc(p, X[VAL], y[VAL])
        if best is None or val > best[1]:
            best = (acc(p, Xt, yt), val, lr)
    return best


stored = {m: LAB["methods"][m]["runs"] for m in LAB["methods"]}
pr100 = {
    "điểm ảnh thô": probe({}, 10, method="pixel"),
    "mạng ngẫu nhiên": probe(init_encoder(0), 10, method="randprobe"),
    "SimCLR 200 bước": probe(sim, 10, lrs=[3e-3, 1e-2]),
}
for name, (te, va, lr) in pr100.items():
    print(f"{name:>16}, 100 nhãn: kiểm tra {pct(te)} (kiểm định {pct(va)}, η = {vn(lr, 4)})")
for name, m in [("điểm ảnh thô", "pixel"), ("mạng ngẫu nhiên", "randprobe")]:
    assert abs(pr100[name][0] - stored[m]["100"][0]["test"]) < 0.005, "khác với demo: hãy kiểm tra lại dữ liệu"
assert pr100["SimCLR 200 bước"][0] > pr100["mạng ngẫu nhiên"][0] + 0.08
print(f"200 bước × 128 ảnh = {pct(200 * 128 / (SIM['steps'] * SIM['batch']), 0)} số ảnh mà SimCLR của demo xem")

Cùng một mạng, cùng khởi tạo, cùng 100 nhãn: chỉ thêm 200 bước học không nhãn, thăm dò tuyến tính đã cao hơn mạng
ngẫu nhiên hơn 10 điểm. Nhưng nó mới quanh mức của hồi quy softmax trên điểm ảnh thô: 200 bước chỉ bằng một phần nhỏ số
bước của demo (in ở dưới), và mỗi bước chỉ có nửa số ảnh. Mục 7 đọc kết quả của bản đầy đủ.

## 7 · Tiền huấn luyện giúp khi nào?

Demo chạy 9 cách học trên cùng mạng, với 10 đến 55.000 nhãn, mỗi ô 3 seed (seed $s$ cố định tập nhãn, lớp đầu ra và
thứ tự lô, nên ba cách học cùng seed nhìn đúng cùng các ảnh). Mỗi ô thử một lưới 6 tốc độ học (mở rộng thêm nếu
$\eta$ tốt nhất nằm ở mép lưới) và chọn $\eta$ trên tập kiểm định. Ở đây ta đọc 5 cách:
- **học từ đầu** và **học từ đầu + tăng cường**: chính phép xoay/co giãn/dịch của SimCLR áp lên ảnh có nhãn ở mỗi bước;
- **SimCLR, thăm dò** và **SimCLR, tinh chỉnh toàn bộ**: bộ mã hóa SimCLR 5.000 bước trên 55.000 ảnh không nhãn;
- **tự mã hóa, tinh chỉnh toàn bộ**: bộ mã hóa của autoencoder sâu ở Bài 19, học tái tạo điểm ảnh trên cùng các ảnh.

### 🤔 Dự đoán trước
Với 100 nhãn, SimCLR hơn học từ đầu vài điểm. Với 55.000 nhãn (gần như cả kho đều có nhãn), tinh chỉnh từ SimCLR hơn
học từ đầu bao nhiêu điểm? Và học từ đầu + tăng cường đứng ở đâu so với SimCLR khi chỉ có 100 nhãn?

In [ ]:
NS = LAB["setup"]["ns"]
NAME = {"scratch": "học từ đầu", "scratchaug": "từ đầu + tăng cường", "simprobe": "SimCLR thăm dò",
        "simfull": "SimCLR toàn bộ", "aefull": "tự mã hóa toàn bộ", "pixel": "điểm ảnh thô",
        "randprobe": "mạng ngẫu nhiên, thăm dò", "aeprobe": "tự mã hóa, thăm dò", "simlora4": "SimCLR + LoRA r 4"}  # fmt: skip


def runs(m, n, key="test"):
    """Kết quả của 3 seed (theo thứ tự seed 0, 1, 2) cho cách học m với n nhãn."""
    return [r[key] for r in LAB["methods"][m]["runs"][str(n)]]


def gain(m, n):
    """Hơn học từ đầu bao nhiêu điểm, theo từng seed (cùng tập nhãn)."""
    return [100 * (a - b) for a, b in zip(runs(m, n), runs("scratch", n))]


def pts(v, digits=1):
    lo, hi = vn(min(v), digits), vn(max(v), digits)
    return lo if lo == hi else f"{lo}–{hi}" if min(v) >= 0 else f"{lo} đến {hi}"


print(
    f"tham số được học: toàn bộ {vn(LAB['methods']['simfull']['params'], 0)} · thăm dò {vn(LAB['methods']['simprobe']['params'], 0)}"
)
for n in NS:
    print(
        f"{vn(n, 0):>6} nhãn: "
        + " · ".join(f"{NAME[m]} {span(runs(m, n), 1)}" for m in ("scratch", "simprobe", "simfull", "scratchaug"))
    )
print()
for n in NS:
    g = {m: gain(m, n) for m in ("simprobe", "simfull", "scratchaug", "aefull")}
    print(f"{vn(n, 0):>6} nhãn, hơn học từ đầu (điểm): " + " · ".join(f"{NAME[m]} {pts(v)}" for m, v in g.items()))

Ở 55.000 nhãn, SimCLR tinh chỉnh toàn bộ hơn học từ đầu từ −0,1 đến 0,2 điểm, tức **trong dao động giữa các seed**:
lợi thế đã biến mất. Lợi thế lớn nhất nằm ở ít nhãn: 5,5–11,3 điểm ở 10 nhãn và 4,0–7,7 điểm ở 100 nhãn (thăm dò),
rồi 0,3–0,5 điểm ở 10.000 nhãn (toàn bộ). He, Girshick & Dollár (2018) thấy điều tương tự khi chuyển từ ImageNet
sang COCO, và ULMFiT (Howard & Ruder, 2018) với 100 văn bản có nhãn bằng học từ đầu với 100 lần nhiều hơn.

Điều bất ngờ hơn là dòng “từ đầu + tăng cường”: không tiền huấn luyện gì, chỉ áp chính phép biến đổi của SimCLR lên
100 ảnh có nhãn, mạng đạt 87,0–90,4%, hơn mọi mô hình tiền huấn luyện. Điều SimCLR dạy ở đây chủ yếu là “xoay nhẹ, dịch
vài điểm ảnh vẫn là chữ số đó”; khi biết trước tính bất biến cần có, đưa thẳng nó vào dữ liệu rẻ hơn. Zoph và cộng sự
(2020) thấy cùng hiện tượng ở quy mô lớn: tăng cường mạnh hơn làm giảm, thậm chí đảo ngược, lợi ích của tiền huấn
luyện ImageNet trên COCO. Tự mã hóa thì không giúp ở mức nhãn nào: tái tạo điểm ảnh không dạy điều mà việc phân loại cần.

Hình dưới vẽ cùng các số. Trái: độ chính xác kiểm tra theo số nhãn (đường qua trung vị 3 seed, vạch đứng là min–max).
Phải: hơn học từ đầu bao nhiêu điểm, theo từng seed; đường xám là 0. Trục ngang là số nhãn theo thang log; các cách
học được dịch sang hai bên một chút để vạch của chúng không chồng lên nhau.

In [ ]:
STYLE = {"scratch": (BLUE, "o", "-"), "scratchaug": (GREEN, "^", "-"), "simprobe": (ORANGE, "s", "-"),
         "simfull": (ORANGE, "D", "--"), "aefull": (GREY, "v", ":")}  # fmt: skip
lx = np.log10(NS)
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.9))
OFFSET = (-0.06, -0.02, 0.02, 0.06)  # dịch ngang từng cách học để các vạch min–max không chồng lên nhau
for o, m in zip(OFFSET, ("scratch", "scratchaug", "simprobe", "simfull")):
    c, mk, ls = STYLE[m]
    v = np.array([runs(m, n) for n in NS]) * 100
    a.plot(lx + o, np.median(v, 1), color=c, marker=mk, ls=ls, label=NAME[m], mfc="white" if ls == "--" else c)
    a.vlines(lx + o, v.min(1), v.max(1), color=c, lw=2)
for o, m in zip(OFFSET, ("simprobe", "simfull", "scratchaug", "aefull")):
    c, mk, ls = STYLE[m]
    g = np.array([gain(m, n) for n in NS])
    b.plot(lx + o, np.median(g, 1), color=c, marker=mk, ls=ls, label=NAME[m], mfc="white" if ls == "--" else c)
    b.vlines(lx + o, g.min(1), g.max(1), color=c, lw=2)
b.axhline(0, color=GREY, lw=1)
for ax, yl in ((a, "đúng trên tập kiểm tra (%)"), (b, "hơn học từ đầu (điểm)")):
    ax.set_xticks(lx, [vn(n, 0) for n in NS])
    ax.set(xlabel="số nhãn (thang log)", ylabel=yl)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(frameon=False, fontsize=8)
a.set_title("Độ chính xác theo số nhãn", fontsize=11)
b.set_title("Lợi thế so với học từ đầu", fontsize=11)
plt.tight_layout()
vn_axes(fig)
plt.show()

Hai điều nữa trong bảng trên. Thứ nhất, ở 10 nhãn, chỉ học lớp đầu ra trên bộ mã hóa SimCLR (45,8–50,4%) tốt hơn
tinh chỉnh toàn bộ (38,4–44,3%): học 235.146 tham số từ 10 ảnh thì dễ quá khớp, và khi lớp đầu ra còn ngẫu nhiên,
gradient của nó làm lệch đặc trưng đã học (Kumar và cộng sự, 2022, phân tích hiện tượng thứ hai, gọi
đó là *feature distortion* và đề xuất thăm dò trước rồi mới tinh chỉnh). Từ 1.000 nhãn thì ngược lại. Thứ hai, thăm dò
SimCLR dừng ở 94,7–94,8% dù có bao nhiêu nhãn, còn mọi cách học toàn bộ mạng đều lên trên 98%: một lớp tuyến tính trên
đặc trưng đóng băng có trần của nó.

## 8 · Thăm dò tuyến tính đo được gì?

Thăm dò tuyến tính hay được dùng làm thước đo “chất lượng biểu diễn”: bộ mã hóa nào cho thăm dò cao hơn thì “tốt hơn”.
Trước khi tin con số đó, hãy đặt mốc.

### 🤔 Dự đoán trước
Một bộ mã hóa **chưa học gì** (khởi tạo He ngẫu nhiên, seed 0), đóng băng, chỉ học lớp đầu ra trên tập “55.000 nhãn”:
đúng bao nhiêu phần trăm trên tập kiểm tra? Đoán bừa là 10%. Và hồi quy softmax trên điểm ảnh thô thì sao?

In [ ]:
pr55 = {
    "mạng ngẫu nhiên": probe(init_encoder(0), 5500, method="randprobe"),
    "điểm ảnh thô": probe({}, 5500, method="pixel"),
}
for (name, (te, va, lr)), m in zip(pr55.items(), ("randprobe", "pixel")):
    print(
        f"{name:>15}, {vn(len(subset(5500, 0)), 0)} nhãn: kiểm tra {pct(te)} (η = {vn(lr, 2)}) · demo, 3 seed: {span(runs(m, 55000))}"
    )
    assert abs(te - stored[m]["55000"][0]["test"]) < 0.003, "khác với demo: hãy kiểm tra lại dữ liệu"
for m in ("aeprobe", "simprobe", "aefull", "simfull"):
    print(f"{NAME[m]:>25}, 55.000 nhãn: {span(runs(m, 55000))}")

Một mạng ngẫu nhiên đã cho 86,07% (85,52–86,35% qua 3 seed), và điểm ảnh thô 92,79%. Lý do: 128 đặc trưng ReLU ngẫu nhiên của ảnh vẫn giữ
phần lớn thông tin của ảnh, và một lớp tuyến tính học từ 53.964 ảnh tìm được cách tách chúng. Thăm dò đo **đặc trưng có
tách được bằng một siêu phẳng hay không**, với một lớp đầu ra có đủ dữ liệu để học; nó không đo mô hình “biết” gì.

Bảng thứ hai cho thấy thăm dò còn có thể xếp hạng ngược. Bộ mã hóa của tự mã hóa cho thăm dò 91,81–92,15%, thấp hơn cả điểm
ảnh thô, và thấp hơn SimCLR (94,7–94,8%) nhiều. Nhưng tinh chỉnh toàn bộ từ nó đạt 98,3–98,4%, ngang SimCLR
(98,4–98,5%). Hewitt & Liang (2019) đề xuất luôn so thăm dò với một **mốc kiểm soát** (một nhiệm vụ mà biểu diễn không
thể “biết”, hay một mạng ngẫu nhiên như ở đây) và chỉ đọc phần chênh lệch.

## 9 · CLIP: ghép ảnh với chú thích

CLIP (Radford và cộng sự, 2021) học từ 400 triệu cặp ảnh–văn bản trên Internet. Một tháp ảnh (ở đây là ViT-B/32, Bài 18)
và một tháp văn bản (một Transformer, Bài 17) đưa ảnh và chú thích vào cùng một không gian. Trong mỗi lô $N$ cặp, gọi
$S_{ij}$ là cosin giữa ảnh $i$ và chú thích $j$; mất mát kéo $N$ cặp đúng lên và đẩy $N^2 - N$ cặp sai xuống:

$$\mathcal L_{\text{CLIP}} = \tfrac12\big(\mathrm{CE}(S/\tau) + \mathrm{CE}(S^\top/\tau)\big)$$

CE lấy cặp đúng $i = j$ làm nhãn theo từng hàng: $S/\tau$ theo chiều ảnh → văn bản, $S^\top/\tau$ theo chiều ngược lại.
Nhiệt độ $\tau$ được học, và bị chặn để $1/\tau$ không vượt 100. Đây là NT-Xent của mục 5, chỉ khác là cặp dương gồm
một ảnh và một câu.

Demo đo một CLIP ViT-B/32 thật (trọng số của OpenAI, 151.277.313 tham số) trên 8 ảnh Commons của Bài 18 và trên MNIST.
Notebook không tải mô hình; nó đọc các cosin đã đo từ `pretrain-clip.json` và tự làm phần còn lại.

In [ ]:
cos = {k: np.array(v) for k, v in CLIP["cos"].items()}  # cos[ngôn ngữ][ảnh i, chú thích j]
SCALE = CLIP["model"]["scale"]  # 1/τ mà CLIP học được


def ce_rows(M):
    """Cross-entropy trung bình khi nhãn của hàng i là cột i."""
    return float(-np.log(np.diag(softmax(M))).mean())


print(f"{CLIP['model']['name']}: {vn(CLIP['model']['params'], 0)} tham số, 1/τ = {vn(SCALE, 2)}")
for lang in ("en", "vi"):
    S = cos[lang]
    right = int((S.argmax(1) == np.arange(8)).sum())
    loss = 0.5 * (ce_rows(SCALE * S) + ce_rows(SCALE * S.T))
    print(
        f"chú thích {lang}: ví dụ “{CLIP['captions'][lang][1]}” · ảnh chọn đúng chú thích {right}/8 · L_CLIP = {vn(loss, 3)}"
    )
print(f"đoán bừa: L = log 8 = {vn(np.log(8), 3)}")
st = CLIP["stats"]
print(
    f"cosin ảnh–chú thích đúng: {vn(st['matched'][0], 2)}–{vn(st['matched'][1], 2)} · sai, trung bình {vn(st['unmatchedMean'], 2)}"
)
print(f"cosin ảnh–ảnh trung bình {vn(st['imgImgMean'], 2)} · chú thích–chú thích {vn(st['txtTxtMean'], 2)}")

Với chú thích tiếng Anh, mỗi ảnh có cosin cao nhất với đúng chú thích của nó, và $1/\tau = 100$ biến khoảng cách nhỏ đó
(0,28–0,33 so với trung bình 0,18) thành xác suất gần 1, nên mất mát gần 0. Với chú thích tiếng Việt, chỉ 5/8 ảnh chọn
đúng: CLIP học từ văn bản Internet, phần lớn là tiếng Anh, và bộ tách từ của nó cắt câu tiếng Việt thành nhiều mảnh
hơn hẳn. Cosin ảnh–ảnh và văn bản–văn bản đều cao hơn cosin ảnh–văn bản: ảnh và chữ nằm ở hai vùng tách biệt của cùng
không gian (khoảng cách giữa hai phương thức, Liang và cộng sự, 2022), nên cosin ảnh–chú thích “đúng” chỉ khoảng 0,3.

Hình dưới vẽ 8 ảnh và hai ma trận cosin (hàng: ảnh, cột: chú thích; ô viền đen là cặp đúng, chấm trắng ở góc ô là chú
thích có cosin cao nhất của mỗi ảnh). Hai ma trận dùng chung nhãn hàng ở bên trái.

In [ ]:
photos = CLIP["photos"]
fig, axes = plt.subplots(1, 8, figsize=(10, 1.7))
for ax, ph in zip(axes, photos):
    ax.imshow(plt.imread(io.BytesIO(base64.b64decode(ph["jpeg"])), format="jpeg"))
    ax.set_title(ph["noun"], fontsize=8)
    ax.axis("off")
plt.show()
print("ảnh từ Wikimedia Commons:", "; ".join(f"{ph['artist']} ({ph['license']})" for ph in photos))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, lang, title in zip(axes, ("en", "vi"), ("chú thích tiếng Anh", "chú thích tiếng Việt")):
    S = cos[lang]
    im = ax.imshow(S, cmap="Blues", vmin=0.1, vmax=0.34)
    for i in range(8):
        ax.add_patch(plt.Rectangle((i - 0.5, i - 0.5), 1, 1, fill=False, ec="black", lw=1.5))
        ax.plot(S[i].argmax() + 0.33, i - 0.3, "o", color="white", mec="black", ms=4)
        for j in range(8):
            ax.text(
                j,
                i,
                vn(S[i, j], 2),
                ha="center",
                va="center",
                fontsize=6.5,
                color="white" if S[i, j] > 0.27 else "black",
            )
    ax.set_xticks(range(8), [ph["noun"] for ph in photos], rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(8), [ph["noun"] for ph in photos] if lang == "en" else [], fontsize=8)
    ax.set_title(f"cosin ảnh–chú thích, {title}", fontsize=10)
fig.colorbar(im, ax=axes, fraction=0.025)
vn_axes(fig)
plt.show()
print("số token của chú thích (en / vi):", CLIP["ntok"]["en"], "/", CLIP["ntok"]["vi"])

### Zero-shot: chọn chú thích gần nhất trong một danh sách do ta đặt

Muốn phân loại 1.000 lớp ImageNet mà không học thêm gì, ta viết 1.000 chú thích theo một **câu mẫu** (“a photo of a
{lớp}.”), lấy vectơ văn bản của chúng, rồi chọn lớp có cosin cao nhất với ảnh. Xác suất là softmax của
$100 \cdot \cos$ trên 1.000 lớp. Demo lưu bảng cosin 8 ảnh × 1.000 lớp (float16) cho 5 câu mẫu và cho trung bình của 7
câu mẫu mà OpenAI chọn. Ô dưới đọc lớp đứng đầu cho từng ảnh. Ảnh hoa hướng dương không có lớp đúng: ImageNet-1k không
có lớp “sunflower”.

In [ ]:
classes = CLIP["classes"]
zs = {
    t["id"]: np.frombuffer(base64.b64decode(CLIP["zs"][t["id"]]), "<f2").astype(np.float64).reshape(8, 1000)
    for t in CLIP["templates"]
}
TEMPL = {
    t["id"]: t["text"][0] if len(t["text"]) == 1 else f"trung bình {len(t['text'])} câu mẫu" for t in CLIP["templates"]
}
for tid in ("bare", "photo", "ens7"):
    top = zs[tid].argmax(1)
    p = softmax(SCALE * zs[tid])
    ok = [int(top[i]) in ph["expect"] for i, ph in enumerate(photos) if ph["expect"]]
    print(
        f"{TEMPL[tid]!r:>22}: đúng {sum(ok)}/{len(ok)} · "
        + ", ".join(f"{classes[top[i]]} {vn(p[i, top[i]], 2)}" for i in range(8))
    )

Câu mẫu đổi đáp án. Chỉ tên lớp trần (“{}”) thì con mèo thành “ruffed grouse” (gà gô); thêm “a photo of a” thì đúng
“tabby cat”. Với hoa hướng dương, CLIP buộc phải chọn một trong 1.000 lớp có sẵn và chọn “cardoon” (một loại atisô):
zero-shot là bài toán tập đóng, nó chỉ trả lời được những gì ta đưa vào danh sách. Thêm “sunflower” làm lớp thứ 1.001:

In [ ]:
for tid in ("bare", "photo", "ens7"):
    s = np.append(zs[tid][7], CLIP["sunflower"][tid][7])
    print(f"{TEMPL[tid]!r:>22}: thêm lớp “sunflower” → xác suất {pct(softmax(SCALE * s[None])[0, -1], 1)}")
print(f"(lớp thứ {vn(len(classes) + 1, 0)})")

Khi danh sách có lớp đúng, CLIP chọn nó với xác suất 68,7–96,1% tùy câu mẫu. Trung bình nhiều câu mẫu thường ổn định
hơn một câu đơn lẻ; đó là cách OpenAI báo kết quả ImageNet.

### CLIP trên chữ số viết tay

Trên ImageNet, CLIP ViT-B/32 đạt 63,2% zero-shot mà không thấy nhãn ImageNet nào (Radford và cộng sự, 2021, bảng 11).
Bài báo cũng báo 51,9% trên MNIST, với câu mẫu `a photo of the number: "{}".`. Demo đo lại trên 10.000 ảnh kiểm tra,
với 6 câu mẫu × 3 cách gọi tên chữ số (chữ số “7”, chữ tiếng Anh “seven”, chữ tiếng Việt “bảy”).

### 🤔 Dự đoán trước
Hồi quy softmax trên điểm ảnh thô (mô hình đơn giản nhất của khóa học) cần bao nhiêu ảnh có nhãn **mỗi lớp** để vượt
CLIP zero-shot trên MNIST: 1, 2, 10, hay 100?

In [ ]:
zsm = CLIP["mnist"]["zeroShot"]
for names in ("digit", "word", "vi"):
    row = [z for z in zsm if z["names"] == names]
    print(f"tên {names:>5}: " + " · ".join(f"{z['template']!r} {pct(z['acc'], 1)}" for z in row))
paper = next(z for z in zsm if z["names"] == "digit" and z["template"] == 'a photo of the number: "{}".')
print(f"câu mẫu của bài báo: {pct(paper['acc'])} (bài báo: {vn(CLIP['mnist']['paper']['zeroShot'], 1)}%; ImageNet: "
      f"{vn(CLIP['mnist']['paper']['imagenetZeroShot'], 1)}%)")  # fmt: skip
M = CLIP["mnist"]
print(f"demo đưa {vn(M['nTest'] + M['nPool'] + M['nVal'], 0)} ảnh MNIST qua CLIP: {vn(M['nTest'], 0)} kiểm tra, "
      f"{vn(M['nPool'], 0)} cho thăm dò, {vn(M['nVal'], 0)} kiểm định")  # fmt: skip
print("số ảnh CLIP đoán là 0…9:", paper["predCounts"])

# Hồi quy softmax trên điểm ảnh thô, học từ k ảnh mỗi lớp: cùng kho 2.000 ảnh và cùng η (chọn trên 500 ảnh kiểm định)
rng = np.random.default_rng(123)
clip_pool = np.sort(np.concatenate([rng.choice(np.flatnonzero(y[:55000] == c), 200, replace=False) for c in range(10)]))
pixel_k = {}
for k in (1, 2):
    for seed in range(3):
        r_ = np.random.default_rng(900 + seed)
        idx = clip_pool[
            np.concatenate([r_.choice(np.flatnonzero(y[clip_pool] == c), k, replace=False) for c in range(10)])
        ]
        ref = next(r for r in CLIP["mnist"]["probe"]["pixel"] if r["k"] == k)["runs"][seed]
        p = fit(
            {"Wh": np.zeros((784, 10), F32), "bh": np.zeros(10, F32)},
            X[idx],
            y[idx],
            ["Wh", "bh"],
            ref["lr"],
            1000,
            seed,
        )
        pixel_k[k, seed] = acc(p, Xt, yt)
        assert abs(pixel_k[k, seed] - ref["test"]) < 0.003, "khác với demo: hãy kiểm tra lại dữ liệu"
    print(f"điểm ảnh thô, {k} ảnh mỗi lớp ({10 * k} ảnh): {span([pixel_k[k, s] for s in range(3)])} (3 seed)")
assert min(pixel_k[2, s] for s in range(3)) > paper["acc"]

Chỉ **2 ảnh mỗi lớp**: hồi quy softmax trên 20 ảnh có nhãn đã hơn CLIP zero-shot (48,25%) ở cả 3 seed. Bài báo CLIP tự
viết điều này: chỉ một đường cơ sở hồi quy logistic trên điểm ảnh thô đã thắng CLIP zero-shot trên MNIST, và dữ liệu
huấn luyện của nó gần như không có ảnh nào giống chữ số viết tay. Câu mẫu đổi kết quả rất mạnh (25,1–48,2% với tên chữ số),
và có câu mẫu khiến CLIP đoán gần như mọi ảnh là cùng một chữ số. Gọi tên bằng tiếng Việt thì gần như đoán bừa (6,6–11,1%).

Biểu diễn của CLIP vẫn hữu ích: thăm dò tuyến tính trên vectơ ảnh của nó (đóng băng) học từ vài ảnh mỗi lớp thì vượt xa
điểm ảnh thô. Hình dưới vẽ cả hai đường theo số ảnh mỗi lớp (3 seed, vạch đứng là min–max); đường gạch là zero-shot.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
for kind, c, mk, label in (
    ("clip", ORANGE, "s", "thăm dò trên vectơ ảnh của CLIP"),
    ("pixel", BLUE, "o", "hồi quy softmax trên điểm ảnh"),
):
    rows = CLIP["mnist"]["probe"][kind]
    ks = np.array([r["k"] for r in rows])
    v = np.array([[q["test"] for q in r["runs"]] for r in rows]) * 100
    ax.plot(ks, np.median(v, 1), color=c, marker=mk, label=label)
    ax.vlines(ks, v.min(1), v.max(1), color=c, lw=2)
ax.axhline(100 * paper["acc"], color=GREY, ls="--", lw=1.5, label="CLIP zero-shot (0 ảnh có nhãn)")
ax.set_xscale("log")
ax.set(xlabel="số ảnh có nhãn mỗi lớp (thang log)", ylabel="đúng trên 10.000 ảnh kiểm tra (%)", ylim=(30, 100))
ax.set_title("MNIST: CLIP zero-shot, thăm dò trên CLIP, và điểm ảnh thô", fontsize=11)
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False, fontsize=8, loc="lower right")
vn_axes(fig)
plt.show()
for kind in ("clip", "pixel"):
    print(
        kind, " · ".join(f"k {r['k']}: {span([q['test'] for q in r['runs']], 1)}" for r in CLIP["mnist"]["probe"][kind])
    )

## 10 · LoRA: đóng băng $W$, học một cập nhật hạng thấp

Tinh chỉnh toàn bộ học lại mọi tham số. Với GPT-3 175B, mỗi việc mới là một bản sao 175 tỉ tham số. **LoRA** (Hu và
cộng sự, 2021) giữ nguyên $W$ và chỉ học một cập nhật có hạng nhỏ $r$, theo quy ước hàng của khóa học:

$$h = xW + \tfrac{\alpha}{r}\, x A B, \qquad A \in \mathbb R^{d_{\text{vào}} \times r}, \;\; B \in \mathbb R^{r \times d_{\text{ra}}}$$

- $A$ khởi tạo ngẫu nhiên $\mathcal N(0, 1/d_{\text{vào}})$, $B = 0$: lúc đầu $AB = 0$, mạng **y hệt** mạng cũ.
- $\Delta W = \tfrac{\alpha}{r} AB$ có hạng tối đa $r$. Nó cần $r(d_{\text{vào}} + d_{\text{ra}})$ tham số thay vì
  $d_{\text{vào}} \cdot d_{\text{ra}}$.
- $\alpha$ là hằng số tỉ lệ. Bài báo đặt $\alpha$ bằng giá trị $r$ đầu tiên họ thử rồi không chỉnh nữa; demo dùng
  $\alpha = r$, nên $\alpha/r = 1$.
- Bài báo viết theo quy ước cột $h = W_0x + BAx$, nên $A$, $B$ của nó là $A^\top$, $B^\top$ ở đây: ma trận chạm vào
  $x$ trước vẫn là ma trận ngẫu nhiên, ma trận còn lại bằng 0.

`forward` và `grads` ở mục 4 đã có sẵn nhánh LoRA (khi `p` có `A0`, `B0`…). Ô dưới thêm LoRA vào cả hai lớp ẩn, đếm số
tham số cần học, kiểm tra rằng $B = 0$ cho đúng mạng cũ, rồi kiểm tra gradient của $A$, $B$ bằng sai phân hữu hạn.

In [ ]:
def add_lora(p, r, seed):
    """LoRA hạng r trên cả hai lớp ẩn: Aᵢ ~ N(0, 1/d_vào), Bᵢ = 0 (như demo; α/r = 1)."""
    rng, q = np.random.default_rng(20_000 + seed), dict(p)
    for i in range(depth(p)):
        a, b = p[f"W{i}"].shape
        q[f"A{i}"] = (rng.standard_normal((a, r)) / np.sqrt(a)).astype(F32)
        q[f"B{i}"] = np.zeros((r, b), F32)
    return q


LORA_KEYS = ["Wh", "bh", "A0", "B0", "A1", "B1"]
FULL_N = 784 * 256 + 256 + 256 * 128 + 128 + 128 * 5 + 5  # mọi tham số, với lớp đầu ra 5 lớp
for r in (1, 2, 4, 8, 16):
    n = sum(r * (a + b) for a, b in itertools.pairwise(SIZES)) + 128 * 5 + 5  # hai cặp A, B + lớp đầu ra mới
    print(f"LoRA r = {r:>2}: {vn(n, 0):>6} tham số cần học ({pct(n / FULL_N, 1)} của {vn(FULL_N, 0)})")
    assert n == TRA["methods"][f"lora{r}"]["params"], "khác với demo: hãy kiểm tra lại dữ liệu"
assert FULL_N == TRA["methods"]["full"]["params"] == 234_501

net = add_lora(add_head(init_encoder(2), 5, 2), 4, 2)
plain = {k: v for k, v in net.items() if k[0] not in "AB"}
print(
    "B = 0: mạng có LoRA cho logit giống hệt mạng cũ:",
    np.array_equal(forward(net, Xt[:64])[1], forward(plain, Xt[:64])[1]),
)
# quy ước cột của bài báo, h = W₀x + B_p A_p x, cho cùng kết quả với A_p = Aᵀ (ngẫu nhiên) và B_p = Bᵀ
x1, W, A, B = Xt[0].astype(np.float64), net["W0"], net["A0"], np.random.default_rng(4).normal(0, 0.1, net["B0"].shape)
assert np.allclose(x1 @ W + (x1 @ A) @ B, W.T @ x1 + B.T @ (A.T @ x1))
# gradient của A, B (B khác 0 để mọi nhánh đều chạy), float64, sai phân trung tâm
g_rng = np.random.default_rng(3)
q64 = {k: v.astype(np.float64) for k, v in net.items()}
q64["B0"], q64["B1"] = g_rng.normal(0, 0.1, q64["B0"].shape), g_rng.normal(0, 0.1, q64["B1"].shape)
xb, yb = Xt[:16].astype(np.float64), yt[:16] % 5
_, g = grads(q64, xb, yb, LORA_KEYS)
worst = 0.0
for k in ("A0", "B0", "A1", "B1"):
    for j in [tuple(g_rng.integers(0, s) for s in q64[k].shape) for _ in range(6)]:
        old = q64[k][j]
        q64[k][j] = old + 1e-6
        lp = grads(q64, xb, yb, [])[0]
        q64[k][j] = old - 1e-6
        lm = grads(q64, xb, yb, [])[0]
        q64[k][j] = old
        num = (lp - lm) / 2e-6
        worst = max(worst, abs(num - g[k][j]) / max(1e-6, abs(num) + abs(g[k][j])))
print(f"gradient của A, B: sai số tương đối lớn nhất {sci(worst)} trên 24 phần tử")
assert worst < 1e-5

LoRA hạng 4 học 6.341 tham số thay vì 234.501, khoảng 37 lần ít hơn. Trong mạng nhỏ này, lớp đầu ra mới (645 tham số)
chiếm một phần đáng kể; trong một Transformer lớn, phần LoRA áp đảo hoàn toàn (mục 11).

### Chuyển việc: từ chữ số 0–4 sang 5–9

Thí nghiệm của demo: một mạng học **có nhãn** đọc chữ số 0–4 (28.076 ảnh của kho), rồi được chỉnh để đọc 5–9 (26.924
ảnh) với một lớp đầu ra 5 lớp mới. Sau đó ta đo hai điều:
- **việc mới:** đúng bao nhiêu trên 4.861 ảnh kiểm tra của 5–9;
- **việc cũ:** gắn lại lớp đầu ra **cũ** lên bộ mã hóa đã chỉnh (LoRA gộp vào $W$), rồi đo trên 5.139 ảnh kiểm tra của
  0–4. Phần tụt xuống là phần mạng đã **quên** (Kirkpatrick và cộng sự, 2017, gọi là *catastrophic forgetting*).

Ô dưới huấn luyện mạng gốc 0–4 của seed 0 như demo: Adam $\eta = 10^{-3}$, 3.000 bước.

In [ ]:
tr_old, tr_new = POOL[y[POOL] < 5], POOL[y[POOL] >= 5]
te_old, te_new = np.flatnonzero(yt < 5), np.flatnonzero(yt >= 5)
BASE = TRA["setup"]["base"]  # η 10⁻³, 3.000 bước
base = fit(
    add_head(init_encoder(0), 5, 0), X[tr_old], y[tr_old], LORA_KEYS[:2] + ["W0", "b0", "W1", "b1"], BASE["lr"],
    BASE["steps"], 0,
)  # fmt: skip
base_acc = acc(base, Xt[te_old], yt[te_old])
print(
    f"{vn(len(tr_old), 0)} ảnh 0–4 · {vn(len(tr_new), 0)} ảnh 5–9 · kiểm tra {vn(len(te_old), 0)} / {vn(len(te_new), 0)}"
)
print(f"mạng gốc: η = {vn(BASE['lr'], 3)}, {vn(BASE['steps'], 0)} bước")
print(f"mạng gốc trên 0–4: {pct(base_acc)} (demo, 3 seed: {span([b['test'] for b in TRA['base']])})")
assert abs(base_acc - TRA["base"][0]["test"]) < 0.003, "khác với demo: hãy kiểm tra lại dữ liệu"

Kết quả có thể lệch demo vài ảnh: 2 luồng BLAS (như Colab) cộng các tích float32 theo thứ tự khác 1 luồng của demo,
và sau hàng nghìn bước Adam, sai khác ở chữ số cuối thành vài ảnh đổi đáp án. Vì vậy các phép `assert` của mục này
có dung sai.

### Ba cách chuyển việc
Ta chỉnh mạng gốc sang 5–9 bằng thăm dò (chỉ học lớp đầu ra mới), LoRA hạng 4 và tinh chỉnh toàn bộ, mỗi cách với
$\eta$ mà demo đã chọn trên kiểm định cho seed 0. Trước khi chạy, thử xếp ba cách theo độ chính xác trên việc mới,
rồi theo mức giữ việc cũ. Có cách nào không quên chút nào không?

In [ ]:
n_new = steps_for(len(tr_new))
chosen = {m: next(r for r in TRA["methods"][m]["runs"]["all"] if r["seed"] == 0) for m in ("probe", "lora4", "full")}


def encoder_of():
    return {k: base[k].copy() for k in ("W0", "b0", "W1", "b1")}  # bản sao bộ mã hóa của mạng gốc


nets = {
    "probe": (add_head(encoder_of(), 5, 100), ["Wh", "bh"]),
    "lora4": (add_lora(add_head(encoder_of(), 5, 100), 4, 0), LORA_KEYS),
    "full": (add_head(encoder_of(), 5, 100), ["Wh", "bh", "W0", "b0", "W1", "b1"]),
}  # lớp đầu ra mới: seed 100, độc lập với lớp đầu ra cũ (như demo)


def merged(p):
    """Bộ mã hóa với LoRA gộp vào: Wᵢ + AᵢBᵢ (α/r = 1)."""
    return {f"W{i}": p[f"W{i}"] + (p[f"A{i}"] @ p[f"B{i}"] if f"A{i}" in p else 0) for i in range(2)} | {
        "b0": p["b0"], "b1": p["b1"]}  # fmt: skip


res = {}
for m, (p, keys) in nets.items():
    fit(p, X[tr_new], y[tr_new] - 5, keys, chosen[m]["lr"], n_new, 0)
    new = acc(p, Xt[te_new], yt[te_new], off=5)
    old = acc(merged(p) | {"Wh": base["Wh"], "bh": base["bh"]}, Xt[te_old], yt[te_old])
    res[m] = (new, old)
    print(f"{m:>6} (η = {vn(chosen[m]['lr'], 3)}): việc mới 5–9 {pct(new)} · việc cũ 0–4 {pct(old)}  "
          f"(demo, seed 0: {pct(chosen[m]['new'])} / {pct(chosen[m]['old'])})")  # fmt: skip
    assert abs(new - chosen[m]["new"]) < 0.008 and abs(old - chosen[m]["old"]) < 0.02, "khác với demo"
assert res["probe"][1] == base_acc and res["lora4"][1] > res["full"][1] and res["full"][0] > res["lora4"][0]

- **Thăm dò** không quên gì: bộ mã hóa đứng yên, nên với lớp đầu ra cũ, việc cũ cho đúng con số của mạng gốc. Nhưng
  đặc trưng học cho 0–4 chỉ đưa việc mới lên 94,69%.
- **Tinh chỉnh toàn bộ** học việc mới tốt nhất, và quên nhiều nhất.
- **LoRA hạng 4** nằm giữa: kém tinh chỉnh toàn bộ chừng nửa điểm ở việc mới, giữ việc cũ tốt hơn vài điểm.

Một seed chưa đủ để kết luận. Hình dưới vẽ mọi cách của demo ở mức “tất cả ảnh”, 3 seed mỗi cách: trục ngang là việc
mới, trục dọc là việc cũ. Góc trên bên phải là tốt nhất.

In [ ]:
METHODS = [("probe", "thăm dò", GREY, "o"), ("lora1", "LoRA r 1", GREEN, "v"), ("lora4", "LoRA r 4", GREEN, "s"),
           ("lora16", "LoRA r 16", GREEN, "D"), ("full", "toàn bộ", ORANGE, "^")]  # fmt: skip
fig, ax = plt.subplots(figsize=(7, 4.2))
for m, label, c, mk in METHODS:
    rs = TRA["methods"][m]["runs"]["all"]
    ax.scatter([100 * r["new"] for r in rs], [100 * r["old"] for r in rs], color=c, marker=mk, s=60, label=label,
               edgecolor="black", lw=0.6, zorder=3)  # fmt: skip
ctrl = TRA["headControl"]["runs"]
ax.scatter([100 * r["new"] for r in ctrl], [100 * r["old"] for r in ctrl], color="white", edgecolor=ORANGE, marker="^",
           s=60, lw=1.5, label="toàn bộ, lớp đầu ra mới = khởi tạo của lớp cũ", zorder=3)  # fmt: skip
ax.set(xlabel="việc mới: đúng trên 5–9 (%)", ylabel="việc cũ: đúng trên 0–4 (%)")
ax.set_title("Học việc mới thì quên việc cũ (tất cả ảnh 5–9, 3 seed)", fontsize=11)
ax.grid(alpha=0.3)
ax.legend(frameon=False, fontsize=8, loc="lower left")
vn_axes(fig)
plt.show()
for m in ("probe", "lora1", "lora2", "lora4", "lora8", "lora16", "full", "scratch"):
    rs = TRA["methods"][m]["runs"]["all"]
    old = f" · việc cũ {span([r['old'] for r in rs], 1)}" if "old" in rs[0] else ""
    print(f"{m:>7} ({vn(TRA['methods'][m]['params'], 0):>7} tham số): việc mới {span([r['new'] for r in rs], 1)}{old}")
print(
    f"toàn bộ, lớp đầu ra mới = khởi tạo của lớp cũ: việc mới {span([r['new'] for r in ctrl], 1)} · việc cũ {span([r['old'] for r in ctrl], 1)}"
)
gap = {m: [100 * (f["new"] - r["new"]) for f, r in zip(TRA["methods"]["full"]["runs"]["all"], TRA["methods"][m]["runs"]["all"])]
       for m in ("lora2", "lora4")}  # fmt: skip


def old_of(ms):
    """Việc cũ (0–4) của mọi seed của các cách học ms, ở mức tất cả ảnh."""
    return [r["old"] for m in ms for r in TRA["methods"][m]["runs"]["all"]]


print(f"LoRA r 2–4 kém toàn bộ ở việc mới (cùng seed): {pts(gap['lora2'] + gap['lora4'], 2)} điểm")
print(f"việc cũ: LoRA r 2–4 {span(old_of(['lora2', 'lora4']))} · toàn bộ {span(old_of(['full']))}"
      f" · seed tệ nhất của r 8 và r 16: {span([min(old_of(['lora8'])), min(old_of(['lora16']))])}")  # fmt: skip

Qua 3 seed, thứ tự giữ nguyên:
- **Việc mới:** thăm dò 94,5–95,0% < LoRA r 4 98,1–98,2% < toàn bộ 98,7–98,8%. LoRA hạng 2–4 kém tinh chỉnh
  toàn bộ 0,55–1,38 điểm, và khoảng cách thu hẹp khi hạng tăng.
- **Việc cũ:** LoRA hạng 2–4 giữ 95,25–98,62%, tinh chỉnh toàn bộ chỉ 80,17–94,82%. Không cách nào chỉnh bộ mã hóa mà
  không quên chút nào; chỉ thăm dò, vốn không chạm vào bộ mã hóa, là không quên.
- **Hạng 8–16:** học việc mới gần bằng toàn bộ, nhưng seed tệ nhất quên nhiều ngang tinh chỉnh toàn bộ (việc cũ
  còn 78,38–86,55%). Mức quên dao
  động giữa các seed rất lớn, vì $\eta$ được chọn cho việc mới chứ không để giảm quên.
- **Một chi tiết nhỏ đổi mức quên:** nếu lớp đầu ra mới bắt đầu từ đúng giá trị khởi tạo của lớp đầu ra cũ (tam giác
  rỗng), tinh chỉnh toàn bộ học việc mới như thường nhưng việc cũ chỉ còn 43,1–55,8%. Con số quên của một lần chạy
  không phải hằng số của phương pháp.

Biderman và cộng sự (2024) thấy cùng hình ảnh ở các mô hình ngôn ngữ lớn: “LoRA học ít hơn và quên ít hơn”. Ít tham số
hơn thì quên ít hơn, không phải không quên.

## 11 · Cập nhật của tinh chỉnh toàn bộ có hạng thấp không?

LoRA dựa trên một giả thuyết: cập nhật $\Delta W$ khi chỉnh cho việc mới có “hạng nội tại” thấp (Hu và cộng sự, 2021,
lấy cảm hứng từ Aghajanyan và cộng sự, 2020). Ta kiểm tra trên chính tinh chỉnh toàn bộ vừa chạy: phân rã giá trị kỳ
dị $\Delta W_1 = W_1^{\text{sau}} - W_1^{\text{trước}} = U\,\mathrm{diag}(\sigma)\,V^\top$, với
$\sigma_1 \ge \sigma_2 \ge \dots$, rồi tính **hạng hiệu dụng**

$$\operatorname{erank}(\Delta W) = \exp\Big(-\sum_k p_k \log p_k\Big), \qquad p_k = \frac{\sigma_k^2}{\sum_j \sigma_j^2}$$

Nó bằng $r$ nếu cập nhật trải đều trên $r$ hướng, và bằng 1 nếu chỉ có một hướng. Demo dùng tỉ lệ năng lượng
$\sigma_k^2$; định nghĩa gốc của Roy & Vetterli (2007) dùng $p_k = \sigma_k / \sum_j \sigma_j$ và cho số lớn hơn.
Ta in cả hai.

### 🤔 Dự đoán trước
$\Delta W_1$ của tinh chỉnh toàn bộ có cỡ 784 × 256, nên hạng tối đa là 256. Hạng hiệu dụng của nó gần 4 (như LoRA
hạng 4), khoảng 50, hay gần 256?

In [ ]:
def erank(dW, power=2):
    """Hạng hiệu dụng exp(H(p)) với p ∝ σᵏ (k = 2: tỉ lệ năng lượng như demo; k = 1: Roy & Vetterli 2007)."""
    s = np.linalg.svd(np.asarray(dW, np.float64), compute_uv=False)
    p = s**power / (s**power).sum()
    p = p[p > 0]
    return float(np.exp(-(p * np.log(p)).sum())), s


full_net, lora_net = nets["full"][0], nets["lora4"][0]
dW = {"toàn bộ": [full_net[f"W{i}"].astype(np.float64) - base[f"W{i}"] for i in range(2)],
      "LoRA r 4": [lora_net[f"A{i}"].astype(np.float64) @ lora_net[f"B{i}"] for i in range(2)]}  # fmt: skip
spec = {}
for name, mats in dW.items():
    for i, M in enumerate(mats):
        er, s = erank(M)
        spec[name, i] = (er, s)
        rel = np.linalg.norm(M) / np.linalg.norm(base[f"W{i}"])
        print(f"{name:>8}, ΔW{i + 1} {M.shape[0]}×{M.shape[1]}: hạng hiệu dụng {vn(er, 1)} (Roy–Vetterli: {vn(erank(M, 1)[0], 1)})"
              f" · ‖ΔW‖/‖W‖ = {vn(rel, 2)}")  # fmt: skip
full_runs = TRA["methods"]["full"]["runs"]["all"]
print(
    f"demo, tinh chỉnh toàn bộ, 3 seed: ΔW₁ {pts([r['erank0'] for r in full_runs])} · ΔW₂ {pts([r['erank1'] for r in full_runs])}"
)
assert 30 < spec["toàn bộ", 0][0] < 70 and spec["LoRA r 4", 0][0] <= 4 + 1e-9

Không phải hạng thấp: cập nhật của tinh chỉnh toàn bộ dùng vài chục hướng (43–59 ở lớp đầu qua 3 seed), còn của LoRA
hạng 4 tối đa 4 hướng, theo định nghĩa. Biderman và cộng sự (2024) đo được điều tương tự ở mô hình ngôn ngữ: cập nhật
của tinh chỉnh toàn bộ có hạng gấp 10–100 lần các cấu hình LoRA thông dụng.

Hình dưới vẽ các giá trị kỳ dị $\sigma_k$ của $\Delta W_1$ (trục dọc thang log): tinh chỉnh toàn bộ (tròn xanh) giảm
dần qua cả trăm hướng; LoRA hạng 4 (vuông cam) chỉ có 4 giá trị, rồi bằng 0.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.4))
s_full, s_lora = spec["toàn bộ", 0][1], spec["LoRA r 4", 0][1][:4]
ax.plot(np.arange(1, len(s_full) + 1), s_full, "o-", color=BLUE, ms=3, lw=1, label="tinh chỉnh toàn bộ")
ax.plot(np.arange(1, 5), s_lora, "s", color=ORANGE, ms=7, label="LoRA hạng 4 (các σ còn lại bằng 0)")
ax.set_yscale("log")
ax.set(xlabel="hướng thứ k", ylabel="σₖ của ΔW₁", xlim=(0, 257))
ax.set_title("Giá trị kỳ dị của cập nhật ΔW₁ (784 × 256)", fontsize=11)
ax.grid(alpha=0.3)
ax.legend(frameon=False, fontsize=9)
vn_axes(fig)
plt.show()
print(
    f"10 hướng đầu giữ {pct((s_full[:10] ** 2).sum() / (s_full**2).sum(), 1)} năng lượng của ΔW₁ toàn bộ; 64 hướng đầu giữ {pct((s_full[:64] ** 2).sum() / (s_full**2).sum(), 1)}"
)

Hạng hiệu dụng 45 không có nghĩa là cần cả 45 hướng. 10 hướng đầu đã giữ hơn nửa năng lượng của $\Delta W_1$. Mục sau
đo trực tiếp xem cần bao nhiêu hướng để giữ độ chính xác. Một điều nữa: cập nhật của LoRA không hề “nhỏ”; tính theo
chuẩn, nó còn lớn hơn cập nhật của tinh chỉnh toàn bộ ($\lVert\Delta W\rVert/\lVert W\rVert$ ở trên). Nó chỉ dồn vào
ít hướng.

## 12 · Giữ $k$ hướng lớn nhất của $\Delta W$

Demo lưu mạng tinh chỉnh toàn bộ của seed 0 để trình duyệt dựng lại nó với bất kỳ $k$ nào:
- $W$ của mạng gốc 0–4, dạng int8 với một hệ số tỉ lệ float16 cho mỗi cột;
- 64 hướng đầu của $\Delta W$ mỗi lớp, dạng float16: các cột của $U\,\mathrm{diag}(\sigma)$ và của $V$;
- độ lệch đã tinh chỉnh, lớp đầu ra mới và lớp đầu ra cũ.

Mạng ở hạng $k$ dùng $W_k = W + \sum_{j \le k} \sigma_j u_j v_j^\top$, tức chỉ giữ $k$ hướng lớn nhất của cập nhật. Ô dưới
giải nén các mảng, dựng lại mạng với $k$ từ 0 đến 64, đo cả việc mới (lớp đầu ra mới) lẫn việc cũ (lớp đầu ra cũ) trên
toàn bộ tập kiểm tra, và so với số mà demo lưu.

In [ ]:
def unpack(t):
    """Một mảng của demo: float16, hoặc int8 nhân hệ số float16 của từng cột (base64, little-endian)."""

    def raw(key, dt):
        return np.frombuffer(base64.b64decode(t[key]), dt).astype(np.float64)

    if "f16" in t:
        return raw("f16", "<f2").reshape(t["shape"])
    return raw("i8", np.int8).reshape(t["shape"]) * raw("scale", "<f2")


W_ship = [unpack(t) for t in SHIP["W"]]
b_ship = [unpack(t) for t in SHIP["b"]]
US, V = [unpack(t) for t in SHIP["US"]], [unpack(t) for t in SHIP["V"]]
head_new, head_old = [(unpack(SHIP[h]["W"]), unpack(SHIP[h]["b"])) for h in ("head", "oldHead")]
print("W:", [w.shape for w in W_ship], "· U·diag(σ):", [u.shape for u in US], "· V:", [v.shape for v in V])
print("σ đầu tiên của ΔW₁ (độ dài cột của U·diag(σ)):", " · ".join(vn(v, 3) for v in np.linalg.norm(US[0], axis=0)[:4]))
print("     … demo lưu, chưa làm tròn float16:", " · ".join(vn(v, 3) for v in full_runs[0]["sv0"][:4]))


def rank_k_acc(k):
    """Độ chính xác của mạng hạng k: việc mới (5–9, lớp đầu ra mới) và việc cũ (0–4, lớp đầu ra cũ)."""
    Ws = [W_ship[i] + US[i][:, :k] @ V[i][:, :k].T for i in range(2)]
    out = []
    for idx, (Wh, bh), off in ((te_new, head_new, 5), (te_old, head_old, 0)):
        h = Xt[idx].astype(np.float64)
        for i in range(2):
            h = np.maximum(h @ Ws[i] + b_ship[i], 0)
        out.append(float(((h @ Wh + bh).argmax(1) + off == yt[idx]).mean()))
    return out


shipped = TRA["shipped"]
trunc = {k: rank_k_acc(k) for k in shipped["k"]}
for j, k in enumerate(shipped["k"]):
    new, old = trunc[k]
    print(
        f"k = {k:>2}: việc mới {pct(new)} · việc cũ {pct(old)}   (demo: {pct(shipped['new'][j])} / {pct(shipped['old'][j])})"
    )
    assert abs(new - shipped["new"][j]) < 3e-4 and abs(old - shipped["old"][j]) < 3e-4, "khác với demo"

Mọi con số trùng với demo. Với $k = 0$ (bộ mã hóa gốc, nhưng độ lệch và lớp đầu ra đã tinh chỉnh), việc mới chỉ còn
khoảng một nửa: lớp đầu ra mới được học cùng với $\Delta W$, nên không dùng được nếu thiếu nó. Thêm 4 hướng đã lên 90,99%;
16 hướng lên 98,15%, gần bằng mạng đầy đủ. Đồng thời việc cũ tụt dần: mỗi hướng thêm vào giúp việc mới và làm hỏng việc
cũ một chút.

Hình dưới vẽ hai đường theo $k$ (trục ngang theo thang log, $k = 0$ đặt ở đầu trục): việc mới (tròn xanh) và việc cũ
(vuông cam) của mạng seed 0, cùng mức của LoRA hạng 4 học trực tiếp (dải xanh lá, 3 seed).

In [ ]:
ks = np.array(shipped["k"])
xk = np.where(ks == 0, 0.5, ks)  # k = 0 đặt ở 0,5 trên trục log
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(xk, [100 * trunc[k][0] for k in ks], "o-", color=BLUE, label="việc mới 5–9 (lớp đầu ra mới)")
ax.plot(xk, [100 * trunc[k][1] for k in ks], "s--", color=ORANGE, label="việc cũ 0–4 (lớp đầu ra cũ)")
lora4_new = [100 * r["new"] for r in TRA["methods"]["lora4"]["runs"]["all"]]
ax.axhspan(min(lora4_new), max(lora4_new), color=GREEN, alpha=0.35, label="LoRA hạng 4, học trực tiếp (việc mới)")
ax.set_xscale("log")
ax.set_xticks(xk, [str(k) for k in ks], fontsize=8)
ax.xaxis.set_minor_locator(NullLocator())
ax.set(xlabel="số hướng k của ΔW được giữ", ylabel="đúng trên tập kiểm tra (%)", ylim=(45, 101))
ax.set_title("Mạng tinh chỉnh toàn bộ (seed 0), giữ k hướng lớn nhất của ΔW", fontsize=11)
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False, fontsize=8, loc="lower right")
vn_axes(fig)
plt.show()


def at(k):
    """Việc mới của 3 seed khi giữ k hướng (demo cắt ΔW của từng seed theo cùng cách)."""
    return [next(r["new"] for r in t["rows"] if r["k"] == k) for t in TRA["trunc"]["full"]]


print(
    f"3 seed của demo: k = 4 → {span(at(4))} · k = 16 → {span(at(16))} · đầy đủ {span([t['all'] for t in TRA['trunc']['full']])}"
)
print(f"LoRA hạng 4, học trực tiếp: {span([r['new'] for r in TRA['methods']['lora4']['runs']['all']])}")

Trên cả 3 seed, giữ 16 hướng lớn nhất cho 98,05–98,17% trên việc mới, ngang LoRA hạng 4 học trực tiếp (98,11–98,17%)
và chỉ kém mạng đầy đủ chừng nửa điểm: đó là lý do một hạng nhỏ thường là đủ. Nhưng 4 hướng cắt ra *sau khi học* chỉ
cho 84,12–92,61%, kém xa LoRA hạng 4: học thẳng trong hạng 4 tìm được 4 hướng tốt hơn 4 hướng lớn nhất của một cập nhật
không bị giới hạn hạng. Shuttleworth và cộng sự (2024) cũng thấy LoRA và tinh chỉnh toàn bộ có thể đạt cùng độ chính
xác mà học những cập nhật có cấu trúc khác hẳn nhau.

## 13 · LoRA ở quy mô lớn: đếm tham số và bộ nhớ

Một adapter LoRA hạng $r$ trên ma trận $d_{\text{vào}} \times d_{\text{ra}}$ có $r(d_{\text{vào}} + d_{\text{ra}})$ tham
số. Bài báo LoRA thử nghiệm chính trên GPT-3 175B: 96 khối, $d_{\text{model}} = 12.288$, LoRA trên $W_q$ và $W_v$ của
mọi khối (192 ma trận $d \times d$). Công thức của bài báo là $|\Theta| = 2 \times \hat L_{\text{LoRA}} \times
d_{\text{model}} \times r$, với $\hat L_{\text{LoRA}}$ là số ma trận được thêm LoRA.

In [ ]:
L_gpt3, d_gpt3, FULL_GPT3 = 96, 12_288, 175_255.8e6  # bảng 4 của bài báo LoRA ghi 175.255,8 triệu tham số
for r in (1, 4, 8):
    n = sum(r * (d_gpt3 + d_gpt3) for _ in range(L_gpt3) for _ in ("W_q", "W_v"))
    assert n == 2 * (2 * L_gpt3) * d_gpt3 * r  # công thức |Θ| của bài báo
    print(
        f"r = {r}: {vn(n, 0):>10} tham số · ít hơn {vn(FULL_GPT3 / n, 0)} lần · tệp lưu 16 bit {vn(2 * n / 1e6, 1)} MB"
    )
    assert n == {1: 4_718_592, 4: 18_874_368, 8: 37_748_736}[r]
    print(f"       ≈ {vn(n / 1e6, 1)} triệu")
print(f"cả mô hình ở 16 bit: {vn(2 * FULL_GPT3 / 1e9, 1)} GB")

Đúng các số của bài báo: 4,7 triệu và 37,7 triệu tham số ở hạng 1 và 8 (bảng 4), 18,9 triệu ở hạng 4 (mục 7.1 ghi
“18M”), và tệp lưu cho mỗi việc mới từ 350 GB xuống khoảng 35 MB. Số tham số cần học giảm khoảng 10.000 lần.

Bộ nhớ GPU thì chỉ giảm khoảng 3 lần: bài báo đo 1,2 TB khi tinh chỉnh toàn bộ và 350 GB với LoRA. Lý do nằm ở dòng
cuối: $W$ đóng băng vẫn phải nằm trong bộ nhớ để chạy lượt xuôi và lượt ngược, tức khoảng 350 GB ở 16 bit. LoRA chỉ bỏ
được gradient và hai trạng thái của Adam cho $W$. **QLoRA** (Dettmers và cộng sự, 2023) đi tiếp: lưu $W$ đóng băng bằng
4 bit (kiểu NF4, cộng một lượng nhỏ cho các hằng số lượng tử hóa), giữ LoRA ở 16 bit, và tinh chỉnh được mô hình
65 tỉ tham số trên một GPU 48 GB.

In [ ]:
L, d, ff, V = 80, 8192, 22_016, 32_000  # cấu hình LLaMA 65B: 80 khối, d 8.192, MLP SwiGLU 22.016, từ vựng 32.000
n65 = (
    2 * V * d + L * (4 * d * d + 3 * d * ff) + (2 * L + 1) * d
)  # embedding + đầu ra, 4 ma trận chú ý + 3 ma trận MLP, RMSNorm
print(f"LLaMA 65B: {vn(n65, 0)} tham số")
for name, bits in (("16 bit", 16), ("4 bit NF4 + 0,127 bit hằng số", 4.127)):
    print(f"  W đóng băng ở {name}: {vn(n65 * bits / 8 / 1e9, 1)} GB")
assert n65 == 65_285_660_672

16 bit cho riêng $W$ của LLaMA 65B đã là khoảng 131 GB, vượt xa một GPU 48 GB; 4 bit đưa nó xuống khoảng 34 GB, phần
còn lại dành cho adapter, trạng thái Adam của adapter và các kích hoạt. Bài báo QLoRA so “hơn 780 GB” cho tinh chỉnh
toàn bộ ở 16 bit với “dưới 48 GB” cho QLoRA.

## 14 · Thử thay đổi

Các ô dưới chạy lại cùng các hàm với tham số khác. Bạn có thể sửa giá trị rồi chạy lại.

### Hạng của LoRA
Mục 10 dùng hạng 4. Ô dưới chạy lại phép chuyển 0–4 → 5–9 với hạng 1 và 16, mỗi hạng với $\eta$ mà demo chọn cho
seed 0, rồi in việc mới, việc cũ và hạng hiệu dụng của $\Delta W_1$. Thêm hạng khác vào `RANKS` (2 hay 8) để thử; với
hạng ngoài 1–16, hãy tự chọn $\eta$ (ví dụ $10^{-2}$).

In [ ]:
RANKS = (1, 16)
for r in RANKS:
    ref = next(q for q in TRA["methods"][f"lora{r}"]["runs"]["all"] if q["seed"] == 0)
    p = fit(add_lora(add_head(encoder_of(), 5, 100), r, 0), X[tr_new], y[tr_new] - 5, LORA_KEYS, ref["lr"], n_new, 0)
    new = acc(p, Xt[te_new], yt[te_new], off=5)
    old = acc(merged(p) | {"Wh": base["Wh"], "bh": base["bh"]}, Xt[te_old], yt[te_old])
    er = erank(p["A0"].astype(np.float64) @ p["B0"])[0]
    print(f"LoRA r = {r:>2} (η = {vn(ref['lr'], 3)}): việc mới {pct(new)} · việc cũ {pct(old)} · hạng hiệu dụng ΔW₁ {vn(er, 1)}"
          f"   (demo, seed 0: {pct(ref['new'])} / {pct(ref['old'])})")  # fmt: skip

Hạng 1 học việc mới kém hạng 4 rõ rệt; hạng 16 gần bằng tinh chỉnh toàn bộ. Việc cũ của hạng 16 thay đổi rất mạnh giữa
các seed (mục 10), nên một lần chạy không nói được nhiều. Hạng hiệu dụng không vượt quá $r$, và nhỏ hơn hẳn $r$ khi
các hướng của $AB$ không đều nhau.

### Nhiệt độ và độ dài của SimCLR
Đổi `TAU` (mục 5) hoặc số bước trong `simclr(steps=..., batch=...)` (mục 6) rồi chạy lại hai mục đó. Nhiệt độ thấp
(0,1) phạt nặng những cặp âm khó nhất; nhiệt độ cao (1) đối xử với mọi cặp âm gần như nhau. Với nhiều bước hơn (1.000
bước, tức 5 lần lâu hơn), thăm dò tuyến tính ở 100 nhãn tiến dần về 80,6–82,0% của demo.

### Một bộ mã hóa ngẫu nhiên khác
Mạng ngẫu nhiên của mục 8 có 128 đặc trưng ReLU. Thử ô dưới với các seed khác: thăm dò cho gần như cùng một con số,
vì một lớp tuyến tính trên đặc trưng ngẫu nhiên đo cấu trúc của dữ liệu nhiều hơn là của mạng.

In [ ]:
for seed in (1, 2):
    te, va, lr = probe(init_encoder(seed), 5500, seed=seed, method="randprobe")
    print(
        f"mạng ngẫu nhiên seed {seed}: thăm dò trên {vn(len(subset(5500, seed)), 0)} nhãn → {pct(te)} (demo: {pct(stored['randprobe']['55000'][seed]['test'])})"
    )

## 15 · Giới hạn và bước tiếp theo

- **Quy mô.** Mọi thí nghiệm ở đây dùng một MLP 235 nghìn tham số trên MNIST. Ở quy mô của GPT hay CLIP, tiền huấn
  luyện không chỉ “giúp khi nhãn ít”: nó là cách thực tế duy nhất để có những năng lực mà không tập nhãn nào dạy nổi.
- **Tăng cường thắng SimCLR ở đây** vì ta biết đúng tính bất biến cần có (xoay, dịch, co giãn chữ số). Với văn bản hay
  ảnh tự nhiên đa dạng, không ai liệt kê hết được, và học từ dữ liệu không nhãn mới là lựa chọn.
- **Thăm dò tuyến tính chỉ là một thước đo.** Luôn đặt mốc mạng ngẫu nhiên và điểm ảnh thô bên cạnh, và nếu được, đo
  cả tinh chỉnh toàn bộ.
- **Quên** phụ thuộc vào $\eta$, số bước, cách khởi tạo lớp đầu ra; LoRA chỉ giảm nó.

Còn một câu hỏi lớn: với một ngân sách tính toán cho trước, nên tiền huấn luyện mô hình lớn cỡ nào, trên bao nhiêu dữ
liệu? Câu trả lời là các quy luật mở rộng, bài tiếp theo.

## Tóm tắt
- Tự giám sát tạo nhãn từ chính dữ liệu: token tiếp theo (GPT), token bị che (BERT, 15%), hai phiên bản của cùng một
  ảnh (SimCLR, NT-Xent), ảnh và chú thích của nó (CLIP, mất mát đối xứng).
- SimCLR giúp khi nhãn ít (100 nhãn: 80,6–82,0% so với 74,1–76,9% học từ đầu); ở 55.000 nhãn lợi thế nằm trong dao
  động giữa các seed. Học từ đầu với chính phép tăng cường của SimCLR còn tốt hơn (87,0–90,4%).
- Thăm dò tuyến tính đo đặc trưng có tách được tuyến tính hay không: một mạng ngẫu nhiên đã 85,52–86,35%, điểm ảnh thô
  92,79%; tự mã hóa thăm dò thấp nhưng tinh chỉnh toàn bộ ngang SimCLR.
- CLIP zero-shot chọn chú thích gần nhất trong một danh sách do ta đặt; câu mẫu đổi đáp án. Trên MNIST nó chỉ 48,25%,
  thua hồi quy softmax trên điểm ảnh thô học từ 2 ảnh mỗi lớp.
- LoRA $h = xW + \tfrac{\alpha}{r}xAB$ ($B = 0$ lúc đầu): hạng 4 học 6.341 thay vì 234.501 tham số, đạt 98,1–98,2% so
  với 98,7–98,8% của tinh chỉnh toàn bộ, và quên ít hơn (việc cũ 95,25–98,62% ở hạng 2–4 so với 80,17–94,82%).
- Cập nhật của tinh chỉnh toàn bộ không có hạng thấp (hạng hiệu dụng 43–59), nhưng 16 hướng lớn nhất đã đủ 98,05–98,17%.
- GPT-3 với LoRA trên $W_q, W_v$: 4.718.592 / 18.874.368 / 37.748.736 tham số ở hạng 1 / 4 / 8; bộ nhớ chỉ giảm khoảng
  3 lần vì $W$ vẫn nằm trong GPU.

## Câu hỏi ôn tập
1. Vì sao BERT không thay mọi token được chọn bằng `[MASK]`?
2. Trong NT-Xent, một phiên bản có bao nhiêu ứng viên, và mất mát bằng bao nhiêu nếu mọi $z$ giống hệt nhau?
3. Một bộ mã hóa mới cho thăm dò tuyến tính cao hơn mạng ngẫu nhiên 2 điểm trên MNIST. Bạn cần thêm những mốc nào trước
   khi nói nó “học được biểu
   diễn tốt”?
4. Vì sao LoRA khởi tạo $B = 0$ chứ không phải cả $A$ lẫn $B$ ngẫu nhiên? Và vì sao không khởi tạo cả hai bằng 0?
5. LoRA giảm số tham số cần học của GPT-3 khoảng 10.000 lần. Vì sao bộ nhớ GPU chỉ giảm khoảng 3 lần?
6. Giữ 4 hướng lớn nhất của $\Delta W$ (sau tinh chỉnh toàn bộ) cho 84,12–92,61%, còn LoRA hạng 4 cho 98,11–98,17%.
   Cả hai đều là cập nhật hạng 4. Vì sao khác nhau?

<details><summary>Gợi ý đáp án</summary>

1. Lúc tinh chỉnh không có `[MASK]` nào. Nếu chỉ phải đoán ở chỗ có `[MASK]`, mô hình có thể bỏ qua biểu diễn của các
   token bình thường; 10% token ngẫu nhiên và 10% giữ nguyên buộc nó giữ biểu diễn tốt cho mọi vị trí.
2. $2B - 1$ ứng viên (mọi phiên bản khác trong lô). Nếu mọi $z$ như nhau, softmax đều trên $2B - 1$ ứng viên nên
   $\ell_i = \log(2B - 1)$; với $B = 4$ là $\log 7$.
3. Một mạng ngẫu nhiên cùng kiến trúc (ở đây 85,52–86,35%) và điểm ảnh thô (92,79%): hơn mạng ngẫu nhiên 2 điểm vẫn
   thấp hơn điểm ảnh thô nhiều. Tốt
   hơn nữa là đo cả tinh chỉnh toàn bộ và học từ đầu với cùng số nhãn.
4. $B = 0$ làm $AB = 0$, nên lúc bắt đầu mạng y hệt mạng đã tiền huấn luyện. Nếu cả hai bằng 0, gradient của $A$ (tỉ lệ
   với $B$) và của $B$ (tỉ lệ với $xA$) đều bằng 0, và LoRA không bao giờ học.
5. $W$ đóng băng vẫn phải nằm trong bộ nhớ (khoảng 350 GB ở 16 bit) để chạy lượt xuôi và ngược. LoRA chỉ bỏ gradient
   và trạng thái Adam của $W$. QLoRA giảm tiếp bằng cách lưu $W$ ở 4 bit.
6. Bốn hướng lớn nhất của một cập nhật được học không giới hạn hạng không phải là bốn hướng tốt nhất cho việc mới:
   phần còn lại của cập nhật (và lớp đầu ra, học cùng nó) dựa vào nhiều hướng khác. LoRA học thẳng trong hạng 4, nên
   lớp đầu ra và 4 hướng thích nghi với nhau.
</details>

## Tìm hiểu thêm
- A. Radford và cộng sự (2018), [Improving Language Understanding by Generative Pre-Training](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf) (GPT).
- J. Howard & S. Ruder (2018), [ULMFiT](https://arxiv.org/abs/1801.06146).
- J. Devlin và cộng sự (2018), [BERT](https://arxiv.org/abs/1810.04805); Y. Liu và cộng sự (2019),
  [RoBERTa](https://arxiv.org/abs/1907.11692).
- J. Yosinski và cộng sự (2014), [How transferable are features in deep neural networks?](https://arxiv.org/abs/1411.1792)
- K. He, R. Girshick & P. Dollár (2018), [Rethinking ImageNet Pre-training](https://arxiv.org/abs/1811.08883);
  B. Zoph và cộng sự (2020), [Rethinking Pre-training and Self-training](https://arxiv.org/abs/2006.06882): tăng cường
  mạnh làm giảm lợi ích của tiền huấn luyện.
- T. Chen và cộng sự (2020), [SimCLR](https://arxiv.org/abs/2002.05709) và [SimCLRv2](https://arxiv.org/abs/2006.10029).
- K. He và cộng sự (2021), [Masked Autoencoders](https://arxiv.org/abs/2111.06377).
- A. Radford và cộng sự (2021), [CLIP](https://arxiv.org/abs/2103.00020); W. Liang và cộng sự (2022),
  [Mind the Gap](https://arxiv.org/abs/2203.02053).
- J. Hewitt & P. Liang (2019), [Designing and Interpreting Probes with Control Tasks](https://arxiv.org/abs/1909.03368);
  J. Wieting & D. Kiela (2019), [No Training Required: Exploring Random Encoders for Sentence Classification](https://arxiv.org/abs/1901.10444).
- A. Kumar và cộng sự (2022), [Fine-Tuning can Distort Pretrained Features](https://arxiv.org/abs/2202.10054) (LP-FT).
- A. Aghajanyan, L. Zettlemoyer & S. Gupta (2020), [Intrinsic Dimensionality Explains Fine-Tuning](https://arxiv.org/abs/2012.13255).
- E. Hu và cộng sự (2021), [LoRA](https://arxiv.org/abs/2106.09685); T. Dettmers và cộng sự (2023),
  [QLoRA](https://arxiv.org/abs/2305.14314).
- D. Biderman và cộng sự (2024), [LoRA Learns Less and Forgets Less](https://arxiv.org/abs/2405.09673);
  R. Shuttleworth và cộng sự (2024), [LoRA vs Full Fine-tuning: An Illusion of Equivalence](https://arxiv.org/abs/2410.21228).
- O. Roy & M. Vetterli (2007), [The Effective Rank](https://zenodo.org/records/40328): hạng hiệu dụng.
- J. Kirkpatrick và cộng sự (2017), [Overcoming catastrophic forgetting in neural networks](https://arxiv.org/abs/1612.00796).

**Bài tiếp theo:** [Bài 23 · Quy luật mở rộng](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/quy-luat-mo-rong/): với một
ngân sách tính toán, nên tiền huấn luyện mô hình lớn cỡ nào, trên bao nhiêu dữ liệu.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")